In [1]:
# Cell 1：依赖、SI 单位宏变量与附件 1 参数
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np

# -----------------------------------------------------------------------------
# 统一约定：求解器内部全部使用 SI 单位
# 电流密度 A/m^2，长度 m，温度 K，压力 Pa，浓度 mol/m^3，水含量 kg/m^3
# -----------------------------------------------------------------------------
F = 96485.0                         # C/mol
R = 8.314                           # J/(mol K)
T_REF = 298.15                      # K
P_REF = 101325.0                    # Pa
T_FREEZE = 273.15                   # K
E_TH = 1.48                         # V，热中性电压
E_OPEN_INITIAL = 0.95               # V，附件 1 给出的初始开路电压

M_H2 = 2.016e-3                     # kg/mol
M_O2 = 31.998e-3                    # kg/mol
M_N2 = 28.014e-3                    # kg/mol
M_WATER = 0.018                     # kg/mol
RHO_ICE = 920.0                     # kg/m^3
RHO_LIQUID = 990.0                  # kg/m^3
LATENT_CONDENSATION = 2.50e6        # J/kg
LATENT_FREEZING = 333600.0          # J/kg

AREA_CELL = 25.0e-4                 # m^2，25 cm^2
L_AGDL = 150.0e-6                   # m
L_ACL = 3.4e-6                      # m
L_PEM = 12.0e-6                     # m
L_CCL = 11.3e-6                     # m
L_CGDL = 150.0e-6                   # m
L_TOTAL = L_AGDL + L_ACL + L_PEM + L_CCL + L_CGDL
L_DIFF_CATHODE = L_CCL + L_CGDL

EPS_AGDL = 0.8
EPS_ACL = 0.3916
EPS_CCL = 0.4207
EPS_CGDL = 0.8
EPS_FLOOR = 1.0e-12

PERM_GDL = 6.2e-12                  # m^2
PERM_CL = 6.2e-13                   # m^2
CONTACT_ANGLE_GDL = 110.0           # deg
CONTACT_ANGLE_CL = 100.0            # deg
SURFACE_TENSION_WATER = 0.075        # N/m，低温范围的一阶近似
LIQUID_RELATIVE_PERMEABILITY_EXPONENT = 3.0
LIQUID_FILM_SATURATION = 0.10         # -，疏水孔壁连通水膜的等效下限
D_LIQUID_CAPILLARY_MAX = 1.0e-6       # m²/s，限制高饱和区的数值刚性

D_H2_REF = 1.10e-4                  # m^2/s
D_O2_REF = 2.20e-5                  # m^2/s
D_WATER_ANODE_REF = 8.69e-5         # m^2/s，aGDL/aCL 水蒸气
D_WATER_CATHODE_REF = 2.48e-5       # m^2/s，cCL/cGDL 水蒸气
DIFFUSIVITY_T_EXPONENT = 1.75
BRUGGEMAN_EXPONENT = 1.5

RHO_PEM = 2150.0                    # kg/m^3
EW_PEM = 1.0                        # kg/mol，1000 g/mol
LAMBDA_INITIAL = 3.0
LAMBDA_MIN = 0.0
LAMBDA_MAX = 22.0
CL_IONOMER_FRACTION = 0.3

ALPHA = 0.30
# 以 -20 ℃为参考，联合标定温度、水合度和冰覆盖对有效交换电流的影响。
# 不添加经验电压偏置，所有修正仍进入 Butler--Volmer 活化损失。
T_J0_REF = 253.15                  # K
J0_REF = 1.119658333e-1            # A/m²，T_J0_REF、lambda=3 时
EA_ACTIVATION = 57411.907           # J/mol
GAMMA_HYDRATION_ACTIVITY = 4.0
HYDRATION_ACTIVITY_MIN = 0.25
HYDRATION_ACTIVITY_MAX = 10.0
R_CONTACT_ASR = 0.01e-4            # ohm m²，0.01 ohm cm²
BETA_ICE_ACTIVE_AREA = 3.5

H_CONVECTION = 40.0                 # W/(m^2 K)
T_INITIAL_DEFAULT = 253.15          # K，附件 1 默认 -20 degC
T_AMBIENT_DEFAULT = 253.15          # K
P_OPERATING = 101325.0              # Pa
Y_H2_IN = 1.0
Y_O2_IN = 0.233
Y_N2_IN = 0.767
Y_WATER_ANODE_IN = 0.0
Y_WATER_CATHODE_IN = 0.0

# 热物性统一写成 (rho [kg/m^3], cp [J/(kg K)], k [W/(m K)])。
# 数值均来自附件 1；气体密度是附件给出的参考密度，不替代理想气体浓度关系。
THERMAL_H2 = (0.089, 14283.0, 0.1672)
THERMAL_O2 = (1.43, 919.31, 0.0246)
THERMAL_N2 = (1.35, 1041.5, 0.0235)
THERMAL_VAPOR = (4.8e-3, 2000.0, 0.10)
THERMAL_LIQUID = (990.0, 4182.0, 0.60)
THERMAL_ICE = (920.0, 2050.0, 2.30)
THERMAL_GDL = (185.0, 545.0, 0.30)
THERMAL_CL = (970.0, 240.0, 0.27)
THERMAL_PEM = (2150.0, 1050.0, 0.24)

# 附件 1 的水传输/相变系数。成对数值按附件中的正向、反向顺序展开。
K_MEMBRANE_TO_VAPOR = 0.001
K_VAPOR_TO_MEMBRANE = 1.0
K_MEMBRANE_LIQUID = 0.5
K_MEMBRANE_ICE = 1.0
K_CONDENSATION = 1.0
K_EVAPORATION = 1.0
K_DESUBLIMATION = 1.0e-4

# 附件 1 未单列液水-冰速率；采用参考文献 [3] Jiao & Li (2009), Table 4。
K_FREEZE_LIQUID = 1.0             # 1/s
K_MELT_ICE = 1.0                  # 1/s

# 启动前已经吹扫：附件 1 第 55--56 行。
LIQUID_INITIAL = 0.0               # kg/m^3
ICE_FRACTION_INITIAL = 0.0           # -


CURRENT_A_CM2_TO_A_M2 = 1.0e4



def air_mass_to_mole_fraction(y_o2: float, y_n2: float) -> tuple[float, float]:
    """把附件 1 的干空气质量分数换算为摩尔分数。"""
    n_o2 = y_o2 / M_O2
    n_n2 = y_n2 / M_N2
    total = n_o2 + n_n2
    return n_o2 / total, n_n2 / total


X_O2_IN, X_N2_IN = air_mass_to_mole_fraction(Y_O2_IN, Y_N2_IN)


def ideal_gas_species_concentration(temperature, pressure, mole_fraction=1.0):
    """由局部 T、p 和摩尔分数计算气体摩尔浓度，单位 mol/m^3。"""
    temperature = np.asarray(temperature, dtype=float)
    pressure = np.asarray(pressure, dtype=float)
    mole_fraction = np.asarray(mole_fraction, dtype=float)
    if np.any(temperature <= 0.0) or np.any(pressure <= 0.0):
        raise ValueError("温度和绝对压力必须为正")
    if np.any((mole_fraction < 0.0) | (mole_fraction > 1.0)):
        raise ValueError("摩尔分数必须位于 [0, 1]")
    return mole_fraction * pressure / (R * temperature)


def C_H2_IN_DEFAULT(temperature, pressure=P_OPERATING):
    """纯氢入口浓度 c_H2(T, p)。"""
    return ideal_gas_species_concentration(temperature, pressure, mole_fraction=1.0)


def C_O2_IN_DEFAULT(temperature, pressure=P_OPERATING):
    """干空气入口氧浓度 c_O2(T, p)，质量分数先换算为摩尔分数。"""
    return ideal_gas_species_concentration(temperature, pressure, mole_fraction=X_O2_IN)

# 气体 Dirichlet 边界保存可调用函数，离散时使用边界处的瞬时 T、p 求值。
BOUNDARY_DEFAULTS = {
    "temperature_left": ("Robin", H_CONVECTION, T_AMBIENT_DEFAULT),
    "temperature_right": ("Robin", H_CONVECTION, T_AMBIENT_DEFAULT),
    "h2_left": ("Dirichlet", C_H2_IN_DEFAULT),
    "o2_right": ("Dirichlet", C_O2_IN_DEFAULT),
    "water_left": ("Dirichlet_mass_fraction", Y_WATER_ANODE_IN),
    "water_right": ("Dirichlet_mass_fraction", Y_WATER_CATHODE_IN),
}

RESIDUAL_TOLERANCE = 1.0e-6


# 数值正则化宽度：仅平滑“蒸汽达到饱和后出现液水”的互补切换。
# 1e-7 kg/m3 远小于模型中的典型水质量浓度，不改变宏观相平衡。
PHASE_SMOOTHING_MASS = 1.0e-5


# 仅建模 MEA 时会漏掉双极板/端板的热惯性；用附件 2 标定的一阶等效热容倍率。
THERMAL_INERTIA_FACTOR = 100.0


In [2]:
# Cell 2：一维 PEMFC 冷启动物理模型
# 求解顺序：相态/孔隙率 -> 传输系数 -> 电压/反应 -> 质量方程 -> 热源/温度方程


def _cell_widths(dx, size):
    """把标量或逐单元宽度统一为长度为 size 的一维数组。"""
    widths = np.asarray(dx, dtype=float)
    if widths.ndim == 0:
        widths = np.full(size, float(widths))
    if widths.shape != (size,) or np.any(widths <= 0.0):
        raise ValueError("dx 必须为正标量或与单元数一致的一维数组")
    return widths


def _weighted_harmonic_face(coefficient, dx):
    """非均匀网格内部面的距离加权调和平均。"""
    coefficient = np.asarray(coefficient, dtype=float)
    if coefficient.ndim != 1 or coefficient.size < 2:
        raise ValueError("coefficient 必须是至少含两个单元的一维数组")
    widths = _cell_widths(dx, coefficient.size)
    left_distance = 0.5 * widths[:-1]
    right_distance = 0.5 * widths[1:]
    denominator = (
        left_distance / np.maximum(coefficient[:-1], EPS_FLOOR)
        + right_distance / np.maximum(coefficient[1:], EPS_FLOOR)
    )
    face = (left_distance + right_distance) / denominator
    blocked = (coefficient[:-1] <= EPS_FLOOR) | (coefficient[1:] <= EPS_FLOOR)
    return np.where(blocked, 0.0, face)


def _diffusive_face_flux(
    field,
    coefficient,
    dx,
    left_value=None,
    right_value=None,
    left_flux=0.0,
    right_flux=0.0,
):
    """构造 N+1 个面通量；正方向统一取 +x。"""
    field = np.asarray(field, dtype=float)
    coefficient = np.asarray(coefficient, dtype=float)
    if field.ndim != 1 or coefficient.shape != field.shape:
        raise ValueError("field 与 coefficient 必须为等长一维数组")
    widths = _cell_widths(dx, field.size)
    face_flux = np.zeros(field.size + 1, dtype=float)
    if field.size > 1:
        face_coefficient = _weighted_harmonic_face(coefficient, widths)
        center_distance = 0.5 * (widths[:-1] + widths[1:])
        face_flux[1:-1] = -face_coefficient * np.diff(field) / center_distance
    if left_value is None:
        face_flux[0] = left_flux
    else:
        face_flux[0] = -coefficient[0] * (field[0] - left_value) / (0.5 * widths[0])
    if right_value is None:
        face_flux[-1] = right_flux
    else:
        face_flux[-1] = -coefficient[-1] * (right_value - field[-1]) / (0.5 * widths[-1])
    return face_flux


def _face_divergence(face_flux, dx):
    """由 N+1 个面通量计算 N 个控制体的散度。"""
    face_flux = np.asarray(face_flux, dtype=float)
    if face_flux.ndim != 1 or face_flux.size < 2:
        raise ValueError("face_flux 必须是一维面通量数组")
    widths = _cell_widths(dx, face_flux.size - 1)
    return np.diff(face_flux) / widths


@dataclass
class GasConservation:
    """H2/O2：d(eps_g*c_k)/dt = -div(N_k) + S_k。"""

    d_h2_ref: float = D_H2_REF
    d_o2_ref: float = D_O2_REF
    t_ref: float = T_REF
    p_ref: float = P_REF
    temperature_exponent: float = DIFFUSIVITY_T_EXPONENT
    porosity_exponent: float = BRUGGEMAN_EXPONENT

    def effective_diffusivity(self, species, temperature, pressure, gas_porosity):
        species_key = species.lower()
        if species_key == "h2":
            d_ref = self.d_h2_ref
        elif species_key == "o2":
            d_ref = self.d_o2_ref
        else:
            raise ValueError("species 必须为 'h2' 或 'o2'")
        temperature = np.asarray(temperature, dtype=float)
        pressure = np.asarray(pressure, dtype=float)
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        if np.any(temperature <= 0.0) or np.any(pressure <= 0.0):
            raise ValueError("温度和绝对压力必须为正")
        return (
            d_ref
            * (temperature / self.t_ref) ** self.temperature_exponent
            * (self.p_ref / pressure)
            * np.maximum(gas_porosity, EPS_FLOOR) ** self.porosity_exponent
        )

    @staticmethod
    def face_flux(
        concentration,
        diffusivity,
        dx,
        left_value=None,
        right_value=None,
        left_flux=0.0,
        right_flux=0.0,
    ):
        return _diffusive_face_flux(
            concentration,
            diffusivity,
            dx,
            left_value=left_value,
            right_value=right_value,
            left_flux=left_flux,
            right_flux=right_flux,
        )

    @staticmethod
    def rhs(concentration, gas_porosity, d_gas_porosity_dt, face_flux, source, dx):
        concentration = np.asarray(concentration, dtype=float)
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        d_gas_porosity_dt = np.asarray(d_gas_porosity_dt, dtype=float)
        source = np.asarray(source, dtype=float)
        return (
            -_face_divergence(face_flux, dx)
            + source
            - concentration * d_gas_porosity_dt
        ) / np.maximum(gas_porosity, EPS_FLOOR)


@dataclass
class WaterConservation:
    """总水 PDE + 蒸汽/液水/冰分配 + 独立冰相控制方程。"""

    molar_mass: float = M_WATER
    rho_pem: float = RHO_PEM
    equivalent_weight: float = EW_PEM
    rho_liquid: float = RHO_LIQUID
    rho_ice: float = RHO_ICE
    k_freeze: float = K_FREEZE_LIQUID
    k_melt: float = K_MELT_ICE
    k_membrane_to_vapor: float = K_MEMBRANE_TO_VAPOR
    k_vapor_to_membrane: float = K_VAPOR_TO_MEMBRANE
    k_membrane_liquid: float = K_MEMBRANE_LIQUID
    k_membrane_ice: float = K_MEMBRANE_ICE

    def membrane_water_content(self, water_mass):
        return (
            self.equivalent_weight
            * np.asarray(water_mass, dtype=float)
            / (self.rho_pem * self.molar_mass)
        )

    def membrane_water_mass(self, lambda_water):
        return (
            self.rho_pem
            * self.molar_mass
            * np.asarray(lambda_water, dtype=float)
            / self.equivalent_weight
        )

    @staticmethod
    def membrane_equilibrium_lambda(water_activity):
        """Springer 吸附等温式；本题气相活度限制在 0--1。"""
        activity = np.clip(np.asarray(water_activity, dtype=float), 0.0, 1.0)
        return np.clip(
            0.043 + 17.81 * activity - 39.85 * activity**2 + 36.0 * activity**3,
            LAMBDA_MIN,
            LAMBDA_MAX,
        )

    def membrane_exchange_flux_to_membrane(
        self,
        vapor_mass,
        gas_porosity,
        liquid_fraction,
        ice_fraction,
        temperature,
        membrane_water_mass,
        interface_width,
    ):
        """
        返回多孔层指向 PEM 的界面扩散通量，单位 kg/(m² s)。

        附件给出的 s⁻¹ 相间转化率乘以界面控制体特征宽度，得到
        Robin 型传质速度。吸水使用蒸汽/液水/冰三项，解吸使用较慢的
        膜水到蒸汽系数；正值始终表示水进入 PEM。
        """
        temperature = float(np.asarray(temperature, dtype=float))
        gas_porosity = max(float(gas_porosity), EPS_FLOOR)
        vapor_mass = max(float(vapor_mass), 0.0)
        liquid_fraction = max(float(liquid_fraction), 0.0)
        ice_fraction = max(float(ice_fraction), 0.0)
        membrane_water_mass = max(float(membrane_water_mass), 0.0)
        interface_width = max(float(interface_width), EPS_FLOOR)

        saturated_bulk_vapor = (
            gas_porosity * float(self.saturated_vapor_density(temperature))
        )
        water_activity = np.clip(
            vapor_mass / max(saturated_bulk_vapor, EPS_FLOOR),
            0.0,
            1.0,
        )
        equilibrium_lambda = float(self.membrane_equilibrium_lambda(water_activity))
        equilibrium_water_mass = float(self.membrane_water_mass(equilibrium_lambda))
        driving_mass = equilibrium_water_mass - membrane_water_mass

        initial_pore_volume = max(
            gas_porosity + liquid_fraction + ice_fraction,
            EPS_FLOOR,
        )
        liquid_saturation = liquid_fraction / initial_pore_volume
        ice_saturation = ice_fraction / initial_pore_volume
        absorption_rate = (
            self.k_vapor_to_membrane
            + self.k_membrane_liquid * liquid_saturation
            + self.k_membrane_ice * ice_saturation
        )
        kinetic_rate = (
            absorption_rate if driving_mass >= 0.0 else self.k_membrane_to_vapor
        )
        return float(kinetic_rate * interface_width * driving_mass)

    @staticmethod
    def electro_osmotic_drag(lambda_water):
        return 2.5 * np.asarray(lambda_water, dtype=float) / 22.0

    @staticmethod
    def membrane_diffusivity(temperature, lambda_water):
        temperature = np.asarray(temperature, dtype=float)
        lambda_water = np.asarray(lambda_water, dtype=float)
        if np.any(temperature <= 0.0):
            raise ValueError("绝对温度必须为正")
        polynomial = (
            2.563
            - 0.33 * lambda_water
            + 0.0264 * lambda_water**2
            - 0.000671 * lambda_water**3
        )
        diffusivity = (
            1.0e-10
            * np.exp(2416.0 * (1.0 / 303.15 - 1.0 / temperature))
            * polynomial
        )
        return np.maximum(diffusivity, 0.0)

    @staticmethod
    def saturation_pressure(temperature):
        """Buck 关系式；零上为水面、零下为冰面饱和蒸气压，单位 Pa。"""
        temperature = np.asarray(temperature, dtype=float)
        t_celsius = temperature - 273.15
        p_water = 611.21 * np.exp(
            (18.678 - t_celsius / 234.5) * t_celsius / (257.14 + t_celsius)
        )
        p_ice = 611.15 * np.exp(
            (23.036 - t_celsius / 333.7) * t_celsius / (279.82 + t_celsius)
        )
        return np.where(t_celsius >= 0.0, p_water, p_ice)

    def saturated_vapor_density(self, temperature):
        """单位气相体积内的饱和水蒸气质量浓度，kg/m^3。"""
        temperature = np.asarray(temperature, dtype=float)
        return self.molar_mass * self.saturation_pressure(temperature) / (R * temperature)

    def phase_state(self, total_water, ice_mass, temperature, initial_porosity):
        """瞬时饱和平衡：总水分为 mv、ml、mi，并同步扣除液水/冰占孔。"""
        total_water, ice_mass, temperature, initial_porosity = np.broadcast_arrays(
            np.maximum(np.asarray(total_water, dtype=float), 0.0),
            np.maximum(np.asarray(ice_mass, dtype=float), 0.0),
            np.asarray(temperature, dtype=float),
            np.asarray(initial_porosity, dtype=float),
        )
        ice_mass = np.minimum(ice_mass, total_water)
        mobile_water = total_water - ice_mass
        ice_fraction = ice_mass / self.rho_ice
        pore_without_liquid = np.maximum(initial_porosity - ice_fraction, EPS_FLOOR)
        vapor_saturation = self.saturated_vapor_density(temperature)
        denominator = np.maximum(1.0 - vapor_saturation / self.rho_liquid, EPS_FLOOR)
        saturation_excess = (
            mobile_water - vapor_saturation * pore_without_liquid
        )
        # 平滑正部函数避免液水初生时 min/max 互补条件导致隐式求解器抖振。
        transition_ratio = np.clip(
            saturation_excess / PHASE_SMOOTHING_MASS,
            0.0,
            1.0,
        )
        transition_excess = PHASE_SMOOTHING_MASS * (
            2.0 * transition_ratio**2 - transition_ratio**3
        )
        positive_excess = np.where(
            saturation_excess <= 0.0,
            0.0,
            np.where(
                saturation_excess >= PHASE_SMOOTHING_MASS,
                saturation_excess,
                transition_excess,
            ),
        )
        liquid_mass = positive_excess / denominator
        liquid_mass = np.minimum(liquid_mass, mobile_water)
        vapor_mass = mobile_water - liquid_mass
        liquid_fraction = liquid_mass / self.rho_liquid
        gas_porosity = np.maximum(
            initial_porosity - liquid_fraction - ice_fraction,
            EPS_FLOOR,
        )
        return {
            "vapor_mass": vapor_mass,
            "liquid_mass": liquid_mass,
            "ice_mass": ice_mass,
            "gas_porosity": gas_porosity,
            "liquid_fraction": liquid_fraction,
            "ice_fraction": ice_fraction,
        }

    @staticmethod
    def vapor_effective_diffusivity(d_ref, temperature, pressure, gas_porosity):
        temperature = np.asarray(temperature, dtype=float)
        pressure = np.asarray(pressure, dtype=float)
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        if np.any(temperature <= 0.0) or np.any(pressure <= 0.0):
            raise ValueError("温度和绝对压力必须为正")
        return (
            np.asarray(d_ref, dtype=float)
            * (temperature / T_REF) ** DIFFUSIVITY_T_EXPONENT
            * (P_REF / pressure)
            * np.maximum(gas_porosity, EPS_FLOOR) ** BRUGGEMAN_EXPONENT
        )

    @staticmethod
    def liquid_dynamic_viscosity(temperature):
        """附件参考关系，kg/(m s)。"""
        temperature = np.asarray(temperature, dtype=float)
        return 2.414e-5 * 10.0 ** (247.8 / np.maximum(temperature - 140.0, 1.0))

    def liquid_capillary_diffusivity(
        self,
        liquid_fraction,
        initial_porosity,
        permeability,
        contact_angle_deg,
        temperature,
    ):
        """Darcy--Leverett 闭合得到的等效液水毛细扩散系数。"""
        liquid_fraction, initial_porosity, permeability, temperature = np.broadcast_arrays(
            np.maximum(np.asarray(liquid_fraction, dtype=float), 0.0),
            np.maximum(np.asarray(initial_porosity, dtype=float), EPS_FLOOR),
            np.maximum(np.asarray(permeability, dtype=float), EPS_FLOOR),
            np.asarray(temperature, dtype=float),
        )
        saturation = np.clip(liquid_fraction / initial_porosity, 0.0, 1.0)
        relative_permeability = np.maximum(
            saturation,
            LIQUID_FILM_SATURATION,
        ) ** LIQUID_RELATIVE_PERMEABILITY_EXPONENT
        leverett_derivative = np.abs(
            1.417 - 4.240 * saturation + 3.789 * saturation**2
        )
        wetting_factor = abs(np.cos(np.deg2rad(float(contact_angle_deg))))
        capillary_scale = (
            SURFACE_TENSION_WATER
            * wetting_factor
            / np.maximum(self.liquid_dynamic_viscosity(temperature), EPS_FLOOR)
            * np.sqrt(permeability / initial_porosity)
        )
        return np.clip(
            capillary_scale * relative_permeability * leverett_derivative,
            0.0,
            D_LIQUID_CAPILLARY_MAX,
        )

    @staticmethod
    def porous_face_flux(
        vapor_mass,
        diffusivity,
        dx,
        left_value=None,
        right_value=None,
        left_flux=0.0,
        right_flux=0.0,
    ):
        return _diffusive_face_flux(
            vapor_mass,
            diffusivity,
            dx,
            left_value=left_value,
            right_value=right_value,
            left_flux=left_flux,
            right_flux=right_flux,
        )

    def membrane_face_flux(
        self,
        water_mass,
        diffusivity,
        proton_current_face,
        lambda_face,
        dx,
        left_value=None,
        right_value=None,
        left_flux=None,
        right_flux=None,
    ):
        # 先计算膜内反扩散；界面给定的是“总水通量”，不应在边界再次叠加拖曳。
        diffusion_flux = _diffusive_face_flux(
            water_mass,
            diffusivity,
            dx,
            left_value=left_value,
            right_value=right_value,
            left_flux=0.0,
            right_flux=0.0,
        )
        proton_current_face = np.asarray(proton_current_face, dtype=float)
        lambda_face = np.asarray(lambda_face, dtype=float)
        if proton_current_face.shape != diffusion_flux.shape or lambda_face.shape != diffusion_flux.shape:
            raise ValueError("质子电流和 lambda_face 必须与面通量同为 N+1 长度")
        drag_flux = (
            self.electro_osmotic_drag(lambda_face)
            * self.molar_mass
            * proton_current_face
            / F
        )
        total_flux = diffusion_flux + drag_flux
        if left_flux is not None:
            total_flux[0] = float(left_flux)
        if right_flux is not None:
            total_flux[-1] = float(right_flux)
        return total_flux

    @staticmethod
    def total_water_rhs(face_flux, reaction_water_source, dx):
        """相变不改变总水，只在独立冰相方程和能量方程中出现。"""
        return -_face_divergence(face_flux, dx) + np.asarray(
            reaction_water_source,
            dtype=float,
        )

    def ice_source(self, liquid_mass, ice_mass, temperature, freezing_temperature=T_FREEZE):
        """S_i>0 表示液水冻结，S_i<0 表示冰融化，单位 kg/(m^3 s)。"""
        liquid_mass = np.maximum(np.asarray(liquid_mass, dtype=float), 0.0)
        ice_mass = np.maximum(np.asarray(ice_mass, dtype=float), 0.0)
        temperature = np.asarray(temperature, dtype=float)
        freezing_temperature = np.asarray(freezing_temperature, dtype=float)
        return np.where(
            temperature < freezing_temperature,
            self.k_freeze * liquid_mass,
            -self.k_melt * ice_mass,
        )

    @staticmethod
    def ice_rhs(ice_source):
        return np.asarray(ice_source, dtype=float)

    @staticmethod
    def condensation_rate(liquid_mass_new, liquid_mass_old, ice_source, dt):
        """无液水对流时 S_cond = d(ml)/dt + S_i；正值冷凝，负值蒸发。"""
        if dt <= 0.0:
            raise ValueError("dt 必须为正")
        return (
            (np.asarray(liquid_mass_new, dtype=float) - np.asarray(liquid_mass_old, dtype=float))
            / dt
            + np.asarray(ice_source, dtype=float)
        )


@dataclass
class ChargeConservation:
    """备注 1 的准稳态电流分配：i_s+i_m=j(t)。"""

    @staticmethod
    def current_distribution(applied_current, x):
        x = np.asarray(x, dtype=float)
        electron_current = np.zeros_like(x)
        proton_current = np.zeros_like(x)
        x_agdl = L_AGDL
        x_acl = x_agdl + L_ACL
        x_pem = x_acl + L_PEM
        x_ccl = x_pem + L_CCL
        mask = x < x_agdl
        electron_current[mask] = applied_current
        mask = (x >= x_agdl) & (x < x_acl)
        xi = (x[mask] - x_agdl) / L_ACL
        electron_current[mask] = applied_current * (1.0 - xi)
        proton_current[mask] = applied_current * xi
        mask = (x >= x_acl) & (x < x_pem)
        proton_current[mask] = applied_current
        mask = (x >= x_pem) & (x < x_ccl)
        xi = (x[mask] - x_pem) / L_CCL
        proton_current[mask] = applied_current * (1.0 - xi)
        electron_current[mask] = applied_current * xi
        mask = x >= x_ccl
        electron_current[mask] = applied_current
        return electron_current, proton_current

    @staticmethod
    def reaction_sources(domain, applied_current):
        return {
            "h2": domain.h2_source(applied_current),
            "o2": domain.o2_source(applied_current),
            "water": domain.water_source(applied_current),
        }

    @staticmethod
    def residual(electron_current, proton_current, applied_current):
        return (
            np.asarray(electron_current, dtype=float)
            + np.asarray(proton_current, dtype=float)
            - np.asarray(applied_current, dtype=float)
        )


@dataclass
class EnergyConservation:
    """热传导 + 电化学热 + 冷凝潜热 + 冻结潜热。"""

    latent_condensation: float = LATENT_CONDENSATION
    latent_freezing: float = LATENT_FREEZING
    thermal_neutral_voltage: float = E_TH

    @staticmethod
    def effective_properties(
        domain,
        gas_porosity,
        liquid_fraction,
        ice_fraction,
        gas_density,
        gas_heat_capacity,
        gas_conductivity,
    ):
        """以附件干层表观物性为基线，加入孔隙流体热容及液水/冰导热修正。"""
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        liquid_fraction = np.asarray(liquid_fraction, dtype=float)
        ice_fraction = np.asarray(ice_fraction, dtype=float)
        gas_density = np.asarray(gas_density, dtype=float)
        gas_heat_capacity = np.asarray(gas_heat_capacity, dtype=float)
        gas_conductivity = np.asarray(gas_conductivity, dtype=float)
        rho_cp = (
            domain.material_density * domain.material_heat_capacity
            + gas_porosity * gas_density * gas_heat_capacity
            + liquid_fraction * THERMAL_LIQUID[0] * THERMAL_LIQUID[1]
            + ice_fraction * THERMAL_ICE[0] * THERMAL_ICE[1]
        )
        conductivity = (
            domain.material_conductivity
            + liquid_fraction * (THERMAL_LIQUID[2] - gas_conductivity)
            + ice_fraction * (THERMAL_ICE[2] - gas_conductivity)
        )
        return rho_cp, np.maximum(conductivity, EPS_FLOOR)

    @staticmethod
    def face_heat_flux(temperature, conductivity, dx, left_flux=0.0, right_flux=0.0):
        return _diffusive_face_flux(
            temperature,
            conductivity,
            dx,
            left_flux=left_flux,
            right_flux=right_flux,
        )

    @staticmethod
    def convection_boundary_flux(
        left_temperature,
        right_temperature,
        ambient_temperature,
        h=H_CONVECTION,
    ):
        """返回沿 +x 的左右边界热通量；左侧向外散热对应负号。"""
        left_flux = -h * (np.asarray(left_temperature) - np.asarray(ambient_temperature))
        right_flux = h * (np.asarray(right_temperature) - np.asarray(ambient_temperature))
        return left_flux, right_flux

    def electrochemical_heat(self, current_density, voltage, active_thickness=L_TOTAL):
        if active_thickness <= 0.0:
            raise ValueError("active_thickness 必须为正")
        return (
            np.asarray(current_density, dtype=float)
            * (self.thermal_neutral_voltage - np.asarray(voltage, dtype=float))
            / active_thickness
        )

    def phase_change_heat(self, ice_source, condensation_rate=0.0):
        return (
            self.latent_freezing * np.asarray(ice_source, dtype=float)
            + self.latent_condensation * np.asarray(condensation_rate, dtype=float)
        )

    @staticmethod
    def temperature_rhs(
        volumetric_heat_capacity,
        heat_flux_face,
        reaction_heat,
        phase_change_heat,
        auxiliary_heat,
        dx,
    ):
        return (
            -_face_divergence(heat_flux_face, dx)
            + np.asarray(reaction_heat, dtype=float)
            + np.asarray(phase_change_heat, dtype=float)
            + np.asarray(auxiliary_heat, dtype=float)
        ) / np.maximum(np.asarray(volumetric_heat_capacity, dtype=float), EPS_FLOOR)


@dataclass
class CellVoltage:
    """冰堵后的可逆电压、活化损失、欧姆损失和浓差损失。"""

    alpha: float = ALPHA
    j0_ref: float = J0_REF
    activation_energy: float = EA_ACTIVATION
    reference_temperature: float = T_J0_REF
    hydration_exponent: float = GAMMA_HYDRATION_ACTIVITY

    @staticmethod
    def reversible_voltage(temperature, p_h2, p_o2):
        temperature = float(np.asarray(temperature, dtype=float))
        p_h2 = max(float(p_h2), EPS_FLOOR)
        p_o2 = max(float(p_o2), EPS_FLOOR)
        return float(
            1.229
            - 8.5e-4 * (temperature - T_REF)
            + R * temperature / (2.0 * F)
            * np.log((p_h2 / P_REF) * np.sqrt(p_o2 / P_REF))
        )

    def exchange_current_density(self, temperature):
        temperature = float(temperature)
        return float(
            self.j0_ref
            * np.exp(
                -self.activation_energy / R
                * (1.0 / temperature - 1.0 / self.reference_temperature)
            )
        )

    def activation_loss(
        self,
        current_density,
        temperature,
        active_area_factor,
        hydration_activity_factor=1.0,
    ):
        j0_effective = (
            self.exchange_current_density(temperature)
            * max(float(active_area_factor), EPS_FLOOR)
            * max(float(hydration_activity_factor), EPS_FLOOR)
        )
        return float(
            R * float(temperature) / (self.alpha * F)
            * np.arcsinh(float(current_density) / (2.0 * j0_effective))
        )

    @staticmethod
    def membrane_conductivity(temperature, lambda_water):
        temperature = np.asarray(temperature, dtype=float)
        lambda_water = np.asarray(lambda_water, dtype=float)
        conductivity = (0.5139 * lambda_water - 0.326) * np.exp(
            1268.0 * (1.0 / 303.15 - 1.0 / temperature)
        )
        return np.maximum(conductivity, EPS_FLOOR)

    def ohmic_loss(self, current_density, temperature_pem, lambda_pem, dx_pem):
        conductivity = self.membrane_conductivity(temperature_pem, lambda_pem)
        widths = _cell_widths(dx_pem, conductivity.size)
        membrane_asr = float(np.sum(widths / conductivity))
        total_asr = membrane_asr + R_CONTACT_ASR
        return float(current_density) * total_asr, membrane_asr, total_asr

    @staticmethod
    def limiting_current(diffusivity_o2, concentration_o2_ccl, dx_cathode):
        diffusivity_o2 = np.asarray(diffusivity_o2, dtype=float)
        widths = _cell_widths(dx_cathode, diffusivity_o2.size)
        transport_resistance = float(
            np.sum(widths / np.maximum(diffusivity_o2, EPS_FLOOR))
        )
        concentration = max(float(np.mean(concentration_o2_ccl)), 0.0)
        return float(4.0 * F * concentration / max(transport_resistance, EPS_FLOOR))

    def concentration_loss(
        self,
        current_density,
        temperature,
        diffusivity_o2,
        concentration_o2_ccl,
        dx_cathode,
    ):
        limiting_current = max(
            self.limiting_current(diffusivity_o2, concentration_o2_ccl, dx_cathode),
            EPS_FLOOR,
        )
        ratio = float(current_density) / limiting_current
        ratio_safe = np.clip(ratio, 0.0, 1.0 - 1.0e-12)
        loss = -R * float(temperature) / (4.0 * F) * np.log1p(-ratio_safe)
        return float(loss), float(limiting_current), bool(ratio >= 1.0)

    def compute(
        self,
        current_density,
        temperature_cell,
        p_h2_acl,
        p_o2_ccl,
        temperature_pem,
        lambda_pem,
        diffusivity_o2_cathode,
        concentration_o2_ccl,
        active_area_factor,
        dx_pem,
        dx_cathode,
    ):
        reversible = self.reversible_voltage(temperature_cell, p_h2_acl, p_o2_ccl)
        active_factor = float(np.clip(active_area_factor, EPS_FLOOR, 1.0))
        lambda_average = max(float(np.mean(lambda_pem)), EPS_FLOOR)
        hydration_activity_factor = float(np.clip(
            (lambda_average / LAMBDA_INITIAL) ** self.hydration_exponent,
            HYDRATION_ACTIVITY_MIN,
            HYDRATION_ACTIVITY_MAX,
        ))
        activation = self.activation_loss(
            current_density,
            temperature_cell,
            active_factor,
            hydration_activity_factor,
        )
        ohmic, membrane_asr, total_asr = self.ohmic_loss(
            current_density,
            temperature_pem,
            lambda_pem,
            dx_pem,
        )
        concentration, limiting_current, transport_limited = self.concentration_loss(
            current_density,
            temperature_cell,
            diffusivity_o2_cathode,
            concentration_o2_ccl,
            dx_cathode,
        )
        voltage = reversible - activation - ohmic - concentration
        return {
            "cell_voltage": float(voltage),
            "reversible_voltage": float(reversible),
            "activation_loss": float(activation),
            "ohmic_loss": float(ohmic),
            "concentration_loss": float(concentration),
            "limiting_current": float(limiting_current),
            "active_area_factor": float(active_factor),
            "exchange_current_density_A_m2": float(
                self.exchange_current_density(temperature_cell)
            ),
            "hydration_activity_factor": hydration_activity_factor,
            "effective_exchange_current_density_A_m2": float(
                self.exchange_current_density(temperature_cell)
                * active_factor
                * hydration_activity_factor
            ),
            "membrane_asr": float(membrane_asr),
            "total_asr": float(total_asr),
            "transport_limited": bool(transport_limited),
        }


In [3]:
# Cell 3：Domain 基类及五个物理域子类
# 设计原则：Domain 只保存静态材料属性、传输模式和区域源项；
# 动态水相分配、气相孔隙率、FVM 通量与守恒方程均由 Cell 2 / 后续求解器负责。

WATER_TRANSPORT_MODES = frozenset({"none", "porous_vapor", "membrane"})


@dataclass
class Domain:
    """一维层域的静态材料属性、变量开关及电化学源项。"""

    name: str
    thickness: float
    porosity: float
    material_density: float
    material_heat_capacity: float
    material_conductivity: float
    permeability: float = 0.0
    contact_angle_deg: Optional[float] = None
    water_transport_mode: str = "none"
    ionomer_fraction: float = 0.0
    transports_h2: bool = False
    transports_o2: bool = False
    conducts_electrons: bool = False
    conducts_protons: bool = False
    is_catalyst_layer: bool = False

    def __post_init__(self):
        if self.thickness <= 0.0:
            raise ValueError(f"{self.name}: thickness 必须为正")
        if not 0.0 <= self.porosity < 1.0:
            raise ValueError(f"{self.name}: porosity 必须位于 [0, 1)")
        if self.material_density <= 0.0:
            raise ValueError(f"{self.name}: material_density 必须为正")
        if self.material_heat_capacity <= 0.0:
            raise ValueError(f"{self.name}: material_heat_capacity 必须为正")
        if self.material_conductivity <= 0.0:
            raise ValueError(f"{self.name}: material_conductivity 必须为正")
        if self.permeability < 0.0:
            raise ValueError(f"{self.name}: permeability 不得为负")
        if self.water_transport_mode not in WATER_TRANSPORT_MODES:
            raise ValueError(
                f"{self.name}: water_transport_mode 必须属于 "
                f"{sorted(WATER_TRANSPORT_MODES)}"
            )
        if not 0.0 <= self.ionomer_fraction <= 1.0:
            raise ValueError(f"{self.name}: ionomer_fraction 必须位于 [0, 1]")
        if self.water_transport_mode == "porous_vapor" and self.porosity <= 0.0:
            raise ValueError(f"{self.name}: porous_vapor 模式要求正孔隙率")

    @property
    def transports_water(self):
        """兼容布尔判断；具体机制由 water_transport_mode 决定。"""
        return self.water_transport_mode != "none"

    @staticmethod
    def _zero_like(value):
        return np.zeros_like(np.asarray(value, dtype=float))

    def h2_source(self, current_density):
        return self._zero_like(current_density)

    def o2_source(self, current_density):
        return self._zero_like(current_density)

    def water_source(self, current_density):
        return self._zero_like(current_density)

    def volume_reaction_current(self, current_density):
        return self._zero_like(current_density)

    def active_area_factor(self, ice_volume_fraction=0.0):
        return np.ones_like(np.asarray(ice_volume_fraction, dtype=float))


class AGDL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_GDL
        super().__init__(
            name="aGDL",
            thickness=L_AGDL,
            porosity=EPS_AGDL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_GDL,
            contact_angle_deg=CONTACT_ANGLE_GDL,
            water_transport_mode="porous_vapor",
            transports_h2=True,
            conducts_electrons=True,
        )


class ACL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_CL
        super().__init__(
            name="aCL",
            thickness=L_ACL,
            porosity=EPS_ACL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_CL,
            contact_angle_deg=CONTACT_ANGLE_CL,
            water_transport_mode="porous_vapor",
            ionomer_fraction=CL_IONOMER_FRACTION,
            transports_h2=True,
            conducts_electrons=True,
            conducts_protons=True,
            is_catalyst_layer=True,
        )

    def h2_source(self, current_density):
        return -np.asarray(current_density, dtype=float) / (2.0 * F * self.thickness)

    def volume_reaction_current(self, current_density):
        return np.asarray(current_density, dtype=float) / self.thickness


class PEM(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_PEM
        super().__init__(
            name="PEM",
            thickness=L_PEM,
            porosity=0.0,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            water_transport_mode="membrane",
            conducts_protons=True,
        )


class CCL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_CL
        super().__init__(
            name="cCL",
            thickness=L_CCL,
            porosity=EPS_CCL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_CL,
            contact_angle_deg=CONTACT_ANGLE_CL,
            water_transport_mode="porous_vapor",
            ionomer_fraction=CL_IONOMER_FRACTION,
            transports_o2=True,
            conducts_electrons=True,
            conducts_protons=True,
            is_catalyst_layer=True,
        )
        self.ice_area_exponent = BETA_ICE_ACTIVE_AREA

    def o2_source(self, current_density):
        return -np.asarray(current_density, dtype=float) / (4.0 * F * self.thickness)

    def water_source(self, current_density):
        return (
            M_WATER
            * np.asarray(current_density, dtype=float)
            / (2.0 * F * self.thickness)
        )

    def volume_reaction_current(self, current_density):
        return np.asarray(current_density, dtype=float) / self.thickness

    def active_area_factor(self, ice_volume_fraction=0.0):
        """建模假设：冰占初始孔隙的比例按 beta 次幂削弱 CCL 有效反应面积。"""
        ice_volume_fraction = np.asarray(ice_volume_fraction, dtype=float)
        available_pore_ratio = np.clip(
            1.0 - ice_volume_fraction / self.porosity,
            0.0,
            1.0,
        )
        return available_pore_ratio**self.ice_area_exponent


class CGDL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_GDL
        super().__init__(
            name="cGDL",
            thickness=L_CGDL,
            porosity=EPS_CGDL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_GDL,
            contact_angle_deg=CONTACT_ANGLE_GDL,
            water_transport_mode="porous_vapor",
            transports_o2=True,
            conducts_electrons=True,
        )


DOMAINS = (AGDL(), ACL(), PEM(), CCL(), CGDL())
DOMAIN_BY_NAME = {domain.name: domain for domain in DOMAINS}

if len(DOMAIN_BY_NAME) != len(DOMAINS):
    raise ValueError("Domain 名称必须唯一")
if not np.isclose(sum(domain.thickness for domain in DOMAINS), L_TOTAL):
    raise ValueError("五个 Domain 的总厚度与 L_TOTAL 不一致")


In [4]:
# Cell 4：一维 cell-centered FVM 网格、区域编号与拓扑
# 3267 个控制体中心保存状态量，3268 个控制面保存通量。
# 空间顺序：face_0 | cell_0 | face_1 | ... | cell_3266 | face_3267

DX_FVM = 0.1e-6                    # m，0.1 μm
BOUNDARY_REGION_ID = -1


@dataclass(frozen=True)
class RegionMeshInfo:

    region_id: int
    domain: Domain
    cell_start: int
    cell_stop: int                 # Python 右开区间
    face_start: int
    face_stop: int                 # Python 右开区间；相邻域共享界面 face
    x_start: float
    x_end: float

    @property
    def name(self):
        return self.domain.name

    @property
    def n_cells(self):
        return self.cell_stop - self.cell_start

    @property
    def n_faces(self):
        return self.face_stop - self.face_start

    @property
    def cell_slice(self):
        return slice(self.cell_start, self.cell_stop)

    @property
    def face_slice(self):
        return slice(self.face_start, self.face_stop)


class FVMGrid1D:
    """五层连续介质的一维均匀 cell-centered 有限体积网格。"""

    def __init__(self, domains=DOMAINS, dx=DX_FVM):
        self.domains = tuple(domains)
        self.dx = float(dx)

        if not self.domains:
            raise ValueError("domains 不能为空")
        if self.dx <= 0.0:
            raise ValueError("dx 必须为正")

        cell_counts = []
        for domain in self.domains:
            count_float = domain.thickness / self.dx
            count = int(np.rint(count_float))
            if count <= 0 or not np.isclose(
                domain.thickness,
                count * self.dx,
                rtol=0.0,
                atol=1.0e-15,
            ):
                raise ValueError(
                    f"{domain.name}: 厚度 {domain.thickness:.6e} m "
                    f"不能被 dx={self.dx:.6e} m 整除"
                )
            cell_counts.append(count)

        self.cell_counts = np.asarray(cell_counts, dtype=np.int32)
        self.n_cells = int(np.sum(self.cell_counts))
        self.n_faces = self.n_cells + 1
        self.length = self.n_cells * self.dx

        if not np.isclose(
            self.length,
            sum(domain.thickness for domain in self.domains),
            rtol=0.0,
            atol=1.0e-15,
        ):
            raise ValueError("离散网格总长度与五层 Domain 总厚度不一致")

        self.face_positions = np.arange(self.n_faces, dtype=float) * self.dx
        self.cell_centers = 0.5 * (
            self.face_positions[:-1] + self.face_positions[1:]
        )
        self.cell_widths = np.diff(self.face_positions)

        self.cell_region_id = np.empty(self.n_cells, dtype=np.int8)
        regions = []
        cell_start = 0

        for region_id, (domain, count) in enumerate(
            zip(self.domains, self.cell_counts)
        ):
            cell_stop = cell_start + int(count)
            self.cell_region_id[cell_start:cell_stop] = region_id
            regions.append(
                RegionMeshInfo(
                    region_id=region_id,
                    domain=domain,
                    cell_start=cell_start,
                    cell_stop=cell_stop,
                    face_start=cell_start,
                    face_stop=cell_stop + 1,
                    x_start=cell_start * self.dx,
                    x_end=cell_stop * self.dx,
                )
            )
            cell_start = cell_stop

        # tuple[RegionMeshInfo] 相当于只读的 vector<struct>，比 vector<map> 更固定可靠。
        self.regions = tuple(regions)
        self.region_by_id = {region.region_id: region for region in self.regions}
        self.region_by_name = {region.name: region for region in self.regions}

        if len(self.region_by_name) != len(self.regions):
            raise ValueError("网格区域名称必须唯一")

        # face f 的左右控制体：内部面为 (f-1, f)，边界外侧用 -1 表示。
        self.face_left_cell = np.arange(self.n_faces, dtype=np.int32) - 1
        self.face_right_cell = np.arange(self.n_faces, dtype=np.int32)
        self.face_right_cell[-1] = -1

        self.face_left_region_id = np.full(
            self.n_faces,
            BOUNDARY_REGION_ID,
            dtype=np.int8,
        )
        self.face_right_region_id = np.full(
            self.n_faces,
            BOUNDARY_REGION_ID,
            dtype=np.int8,
        )

        has_left_cell = self.face_left_cell >= 0
        has_right_cell = self.face_right_cell >= 0
        self.face_left_region_id[has_left_cell] = self.cell_region_id[
            self.face_left_cell[has_left_cell]
        ]
        self.face_right_region_id[has_right_cell] = self.cell_region_id[
            self.face_right_cell[has_right_cell]
        ]

        self.boundary_face_ids = np.array([0, self.n_faces - 1], dtype=np.int32)
        self.interface_face_ids = np.flatnonzero(
            has_left_cell
            & has_right_cell
            & (self.face_left_region_id != self.face_right_region_id)
        ).astype(np.int32)

        # 网格拓扑在构造后不应被求解器原地修改。
        for array in (
            self.cell_counts,
            self.face_positions,
            self.cell_centers,
            self.cell_widths,
            self.cell_region_id,
            self.face_left_cell,
            self.face_right_cell,
            self.face_left_region_id,
            self.face_right_region_id,
            self.boundary_face_ids,
            self.interface_face_ids,
        ):
            array.setflags(write=False)

    def region_info(self, region):
        """按区域名称或整数 region_id 返回 RegionMeshInfo。"""
        if isinstance(region, str):
            return self.region_by_name[region]
        return self.region_by_id[int(region)]

    def cell_slice(self, region):
        return self.region_info(region).cell_slice

    def face_slice(self, region):
        return self.region_info(region).face_slice

    def expand_domain_attribute(self, attribute, dtype=float):
        """把五个 Domain 的标量属性展开成长度为 n_cells 的单元数组。"""
        region_values = np.asarray(
            [getattr(domain, attribute) for domain in self.domains],
            dtype=dtype,
        )
        return region_values[self.cell_region_id]

    def cell_mask(self, attribute):
        """按 Domain 布尔属性生成控制体掩码。"""
        return self.expand_domain_attribute(attribute, dtype=bool)


FVM_GRID = FVMGrid1D()

# 后续 RHS 直接使用的紧凑别名。
N_CELLS = FVM_GRID.n_cells
N_FACES = FVM_GRID.n_faces
X_CELL = FVM_GRID.cell_centers
X_FACE = FVM_GRID.face_positions
DX_CELL = FVM_GRID.cell_widths
REGION_ID = FVM_GRID.cell_region_id
REGION_TABLE = FVM_GRID.regions
MESH_REGION_BY_ID = FVM_GRID.region_by_id
MESH_REGION_BY_NAME = FVM_GRID.region_by_name

# 常用空间掩码和静态材料场；动态气相孔隙率不在这里存储。
POROSITY_DRY_CELL = FVM_GRID.expand_domain_attribute("porosity")
PERMEABILITY_CELL = FVM_GRID.expand_domain_attribute("permeability")
H2_CELL_MASK = FVM_GRID.cell_mask("transports_h2")
O2_CELL_MASK = FVM_GRID.cell_mask("transports_o2")
ELECTRON_CELL_MASK = FVM_GRID.cell_mask("conducts_electrons")
PROTON_CELL_MASK = FVM_GRID.cell_mask("conducts_protons")
CATALYST_CELL_MASK = FVM_GRID.cell_mask("is_catalyst_layer")
POROUS_WATER_CELL_MASK = np.array(
    [DOMAINS[region_id].water_transport_mode == "porous_vapor" for region_id in REGION_ID],
    dtype=bool,
)
MEMBRANE_WATER_CELL_MASK = np.array(
    [DOMAINS[region_id].water_transport_mode == "membrane" for region_id in REGION_ID],
    dtype=bool,
)

# 明确验证题设网格规模、各层划分以及四个材料界面的位置。
EXPECTED_CELL_COUNTS = np.array([1500, 34, 120, 113, 1500], dtype=np.int32)
EXPECTED_INTERFACE_FACES = np.array([1500, 1534, 1654, 1767], dtype=np.int32)

if N_CELLS != 3267 or N_FACES != 3268:
    raise ValueError("FVM 网格必须包含 3267 个控制体和 3268 个控制面")
if not np.array_equal(FVM_GRID.cell_counts, EXPECTED_CELL_COUNTS):
    raise ValueError("五个物理域的控制体数量不符合预期")
if not np.array_equal(FVM_GRID.interface_face_ids, EXPECTED_INTERFACE_FACES):
    raise ValueError("材料界面 face 编号不符合预期")
if not np.allclose(DX_CELL, DX_FVM, rtol=0.0, atol=1.0e-18):
    raise ValueError("网格不是均匀的 0.1 μm 控制体")

# cell 宽度：3267 个
CELL_WIDTH = FVM_GRID.cell_widths

# 内部 face 两侧 cell center 的距离：3266 个
CELL_CENTER_DISTANCE = np.diff(X_CELL)

# 外边界 face 到第一个/最后一个 cell center 的距离
BOUNDARY_HALF_WIDTH = 0.5 * DX_FVM

FACE_AGDL_ACL = FVM_GRID.region_info("aGDL").cell_stop
FACE_ACL_PEM = FVM_GRID.region_info("aCL").cell_stop
FACE_PEM_CCL = FVM_GRID.region_info("PEM").cell_stop
FACE_CCL_CGDL = FVM_GRID.region_info("cCL").cell_stop

FACE_LEFT_BOUNDARY = 0
FACE_RIGHT_BOUNDARY = N_FACES - 1

In [5]:
# Cell 5：冷启动工况、边界条件与初始条件
# 当前闭合：多孔域保存总水和冰质量，蒸汽/液水由 WaterConservation.phase_state() 瞬时分配；
# PEM 中的 total_water 表示吸附态膜水，不参与多孔域的蒸汽/液水相态分配。


@dataclass(frozen=True)
class ColdStartCase:
    """一次冷启动计算使用的运行条件和初始含水参数。"""

    initial_temperature: float = T_INITIAL_DEFAULT
    ambient_temperature: float = T_AMBIENT_DEFAULT
    pressure: float = P_OPERATING
    initial_lambda: float = LAMBDA_INITIAL
    initial_liquid_mass: float = LIQUID_INITIAL
    initial_ice_fraction: float = ICE_FRACTION_INITIAL
    convection_coefficient: float = H_CONVECTION

    def __post_init__(self):
        if self.initial_temperature <= 0.0 or self.ambient_temperature <= 0.0:
            raise ValueError("初始温度和环境温度必须大于 0 K")
        if self.pressure <= 0.0:
            raise ValueError("运行压力必须为正")
        if self.initial_lambda < 0.0:
            raise ValueError("初始膜含水量 lambda 不得为负")
        if self.initial_liquid_mass < 0.0:
            raise ValueError("初始液水质量浓度不得为负")
        if self.initial_ice_fraction < 0.0:
            raise ValueError("初始冰体积分数不得为负")
        if self.convection_coefficient < 0.0:
            raise ValueError("对流换热系数不得为负")

        porous_porosity = POROSITY_DRY_CELL[POROUS_WATER_CELL_MASK]
        if np.any(self.initial_ice_fraction > porous_porosity):
            raise ValueError("初始冰体积分数不能超过任一多孔域的初始孔隙率")


@dataclass(frozen=True)
class ColdStartBoundaryConditions:
    """Cell 7 离散通量时使用的外边界和气体阻隔边界。"""

    ambient_temperature: float
    pressure: float
    convection_coefficient: float
    water_vapor_left: float = 0.0       # kg/m^3，干氢入口
    water_vapor_right: float = 0.0      # kg/m^3，干空气入口
    h2_right_flux: float = 0.0          # mol/(m^2 s)，aCL/PEM 阻隔面
    o2_left_flux: float = 0.0           # mol/(m^2 s)，PEM/cCL 阻隔面

    @classmethod
    def from_case(cls, case):
        return cls(
            ambient_temperature=case.ambient_temperature,
            pressure=case.pressure,
            convection_coefficient=case.convection_coefficient,
        )

    def h2_left_concentration(self, boundary_temperature):
        """左物理边界：纯氢 Dirichlet 浓度，mol/m^3。"""
        return C_H2_IN_DEFAULT(boundary_temperature, self.pressure)

    def o2_right_concentration(self, boundary_temperature):
        """右物理边界：干空气 O2 Dirichlet 浓度，mol/m^3。"""
        return C_O2_IN_DEFAULT(boundary_temperature, self.pressure)


@dataclass(frozen=True)
class InitialConditions:
    """定义在 3267 个控制体中心上的完整初始物理场。"""

    temperature: np.ndarray
    c_h2: np.ndarray
    c_o2: np.ndarray
    total_water: np.ndarray
    ice_mass: np.ndarray

    def __post_init__(self):
        for field_name in (
            "temperature",
            "c_h2",
            "c_o2",
            "total_water",
            "ice_mass",
        ):
            array = np.asarray(getattr(self, field_name), dtype=float)
            if array.shape != (N_CELLS,):
                raise ValueError(f"{field_name} 必须是长度为 {N_CELLS} 的一维数组")
            if not np.all(np.isfinite(array)):
                raise ValueError(f"{field_name} 包含非有限值")
            if field_name != "temperature" and np.any(array < 0.0):
                raise ValueError(f"{field_name} 不得包含负值")
            if field_name == "temperature" and np.any(array <= 0.0):
                raise ValueError("temperature 必须全部大于 0 K")

            array = array.copy()
            array.setflags(write=False)
            object.__setattr__(self, field_name, array)


def membrane_water_mass_from_lambda(lambda_water):
    """由膜含水量 lambda 得到 PEM 吸附水质量浓度，kg/m^3。"""
    lambda_water = np.asarray(lambda_water, dtype=float)
    if np.any(lambda_water < 0.0):
        raise ValueError("lambda_water 不得为负")
    return lambda_water * RHO_PEM * M_WATER / EW_PEM


def build_initial_conditions(case):
    """按物理作用域构造初始场；不把 PEM 初始膜水扩散到其他四层。"""
    temperature = np.full(N_CELLS, case.initial_temperature, dtype=float)
    c_h2 = np.zeros(N_CELLS, dtype=float)
    c_o2 = np.zeros(N_CELLS, dtype=float)
    total_water = np.zeros(N_CELLS, dtype=float)
    ice_mass = np.zeros(N_CELLS, dtype=float)

    # 启动前气体已经充满相应流道侧多孔域；其余区域不保存无物理意义的气体浓度。
    c_h2[H2_CELL_MASK] = C_H2_IN_DEFAULT(
        case.initial_temperature,
        case.pressure,
    )
    c_o2[O2_CELL_MASK] = C_O2_IN_DEFAULT(
        case.initial_temperature,
        case.pressure,
    )

    # 附件给出的初始液水和冰只属于 GDL/CL 多孔域。
    ice_mass[POROUS_WATER_CELL_MASK] = case.initial_ice_fraction * RHO_ICE
    total_water[POROUS_WATER_CELL_MASK] = (
        case.initial_liquid_mass + ice_mass[POROUS_WATER_CELL_MASK]
    )

    # lambda_0=3 只初始化 PEM 吸附态水；不额外假设 CL 离聚物也取 lambda_0=3。
    pem_slice = FVM_GRID.cell_slice("PEM")
    total_water[pem_slice] = membrane_water_mass_from_lambda(case.initial_lambda)

    return InitialConditions(
        temperature=temperature,
        c_h2=c_h2,
        c_o2=c_o2,
        total_water=total_water,
        ice_mass=ice_mass,
    )


DEFAULT_CASE = ColdStartCase()
BOUNDARY_CONDITIONS_DEFAULT = ColdStartBoundaryConditions.from_case(DEFAULT_CASE)
INITIAL_CONDITIONS_DEFAULT = build_initial_conditions(DEFAULT_CASE)
PEM_WATER_INITIAL = float(membrane_water_mass_from_lambda(LAMBDA_INITIAL))

# 全局 face 编号：物理边界与两种气体各自求解区间的阻隔边界。
H2_BOUNDARY_FACES = (FACE_LEFT_BOUNDARY, FACE_ACL_PEM)
O2_BOUNDARY_FACES = (FACE_PEM_CCL, FACE_RIGHT_BOUNDARY)
THERMAL_BOUNDARY_FACES = (FACE_LEFT_BOUNDARY, FACE_RIGHT_BOUNDARY)

if not np.isclose(PEM_WATER_INITIAL, 116.1, rtol=0.0, atol=1.0e-12):
    raise ValueError("lambda_0=3 对应的 PEM 初始吸附水应为 116.1 kg/m^3")


In [6]:
# Cell 6：物理空间场与 solve_ivp 一维 NumPy 状态向量之间的映射
# 动态状态：T(全域)、H2(阳极多孔域)、O2(阴极多孔域)、总水(全域)、冰质量(全域)。

N_T = N_CELLS
N_H2 = int(np.count_nonzero(H2_CELL_MASK))
N_O2 = int(np.count_nonzero(O2_CELL_MASK))
N_MW = N_CELLS
N_MI = N_CELLS

STATE_T = slice(0, N_T)
STATE_H2 = slice(STATE_T.stop, STATE_T.stop + N_H2)
STATE_O2 = slice(STATE_H2.stop, STATE_H2.stop + N_O2)
STATE_MW = slice(STATE_O2.stop, STATE_O2.stop + N_MW)
STATE_MI = slice(STATE_MW.stop, STATE_MW.stop + N_MI)
N_STATE = STATE_MI.stop

STATE_SLICES = {
    "temperature": STATE_T,
    "c_h2": STATE_H2,
    "c_o2": STATE_O2,
    "total_water": STATE_MW,
    "ice_mass": STATE_MI,
}

H2_STATE_CELL_IDS = np.flatnonzero(H2_CELL_MASK).astype(np.int32)
O2_STATE_CELL_IDS = np.flatnonzero(O2_CELL_MASK).astype(np.int32)
ALL_STATE_CELL_IDS = np.arange(N_CELLS, dtype=np.int32)
for cell_ids in (H2_STATE_CELL_IDS, O2_STATE_CELL_IDS, ALL_STATE_CELL_IDS):
    cell_ids.setflags(write=False)

STATE_CELL_IDS = {
    "temperature": ALL_STATE_CELL_IDS,
    "c_h2": H2_STATE_CELL_IDS,
    "c_o2": O2_STATE_CELL_IDS,
    "total_water": ALL_STATE_CELL_IDS,
    "ice_mass": ALL_STATE_CELL_IDS,
}


def _require_full_cell_field(field_name, values):
    values = np.asarray(values, dtype=float)
    if values.shape != (N_CELLS,):
        raise ValueError(f"{field_name} 必须是长度为 {N_CELLS} 的一维数组")
    if not np.all(np.isfinite(values)):
        raise ValueError(f"{field_name} 包含非有限值")
    return values


def pack_state(temperature, c_h2, c_o2, total_water, ice_mass):
    """把五个完整物理场压缩为 solve_ivp 使用的一维状态向量。"""
    temperature = _require_full_cell_field("temperature", temperature)
    c_h2 = _require_full_cell_field("c_h2", c_h2)
    c_o2 = _require_full_cell_field("c_o2", c_o2)
    total_water = _require_full_cell_field("total_water", total_water)
    ice_mass = _require_full_cell_field("ice_mass", ice_mass)

    state = np.empty(N_STATE, dtype=float)
    state[STATE_T] = temperature
    state[STATE_H2] = c_h2[H2_STATE_CELL_IDS]
    state[STATE_O2] = c_o2[O2_STATE_CELL_IDS]
    state[STATE_MW] = total_water
    state[STATE_MI] = ice_mass
    return state


def unpack_state(state):
    """恢复五个完整空间场；H2/O2 在非传输域自动补零。"""
    state = np.asarray(state, dtype=float)
    if state.shape != (N_STATE,):
        raise ValueError(f"状态向量必须是长度为 {N_STATE} 的一维数组")
    if not np.all(np.isfinite(state)):
        raise ValueError("状态向量包含非有限值")

    temperature = state[STATE_T].copy()
    c_h2 = np.zeros(N_CELLS, dtype=float)
    c_o2 = np.zeros(N_CELLS, dtype=float)
    c_h2[H2_STATE_CELL_IDS] = state[STATE_H2]
    c_o2[O2_STATE_CELL_IDS] = state[STATE_O2]
    total_water = state[STATE_MW].copy()
    ice_mass = state[STATE_MI].copy()

    return temperature, c_h2, c_o2, total_water, ice_mass


def pack_rhs(
    d_temperature_dt,
    d_c_h2_dt,
    d_c_o2_dt,
    d_total_water_dt,
    d_ice_mass_dt,
):
    """把 Cell 7 算得的五个完整导数字段压缩为 dy/dt。"""
    d_temperature_dt = _require_full_cell_field("d_temperature_dt", d_temperature_dt)
    d_c_h2_dt = _require_full_cell_field("d_c_h2_dt", d_c_h2_dt)
    d_c_o2_dt = _require_full_cell_field("d_c_o2_dt", d_c_o2_dt)
    d_total_water_dt = _require_full_cell_field(
        "d_total_water_dt",
        d_total_water_dt,
    )
    d_ice_mass_dt = _require_full_cell_field("d_ice_mass_dt", d_ice_mass_dt)

    rhs = np.empty(N_STATE, dtype=float)
    rhs[STATE_T] = d_temperature_dt
    rhs[STATE_H2] = d_c_h2_dt[H2_STATE_CELL_IDS]
    rhs[STATE_O2] = d_c_o2_dt[O2_STATE_CELL_IDS]
    rhs[STATE_MW] = d_total_water_dt
    rhs[STATE_MI] = d_ice_mass_dt
    return rhs


Y0_DEFAULT = pack_state(
    INITIAL_CONDITIONS_DEFAULT.temperature,
    INITIAL_CONDITIONS_DEFAULT.c_h2,
    INITIAL_CONDITIONS_DEFAULT.c_o2,
    INITIAL_CONDITIONS_DEFAULT.total_water,
    INITIAL_CONDITIONS_DEFAULT.ice_mass,
)

if N_STATE != 12948:
    raise ValueError("当前网格的压缩状态向量长度应为 12948")

_roundtrip_fields = unpack_state(Y0_DEFAULT)
for _original, _restored in zip(
    (
        INITIAL_CONDITIONS_DEFAULT.temperature,
        INITIAL_CONDITIONS_DEFAULT.c_h2,
        INITIAL_CONDITIONS_DEFAULT.c_o2,
        INITIAL_CONDITIONS_DEFAULT.total_water,
        INITIAL_CONDITIONS_DEFAULT.ice_mass,
    ),
    _roundtrip_fields,
):
    if not np.array_equal(_original, _restored):
        raise ValueError("初始场 pack/unpack 往返检查失败")
del _roundtrip_fields, _original, _restored


In [7]:
# Cell 7：物理对象实例化、派生场诊断核与总装 RHS
# compute_model_fields() 是唯一耦合入口：Cell 8 求导、Cell 9 后处理都调用它。

GAS_MODEL = GasConservation()
WATER_MODEL = WaterConservation()
CHARGE_MODEL = ChargeConservation()
ENERGY_MODEL = EnergyConservation()
VOLTAGE_MODEL = CellVoltage()

AGDL_DOMAIN = DOMAIN_BY_NAME["aGDL"]
ACL_DOMAIN = DOMAIN_BY_NAME["aCL"]
PEM_DOMAIN = DOMAIN_BY_NAME["PEM"]
CCL_DOMAIN = DOMAIN_BY_NAME["cCL"]
CGDL_DOMAIN = DOMAIN_BY_NAME["cGDL"]

AGDL_SLICE = FVM_GRID.cell_slice("aGDL")
ACL_SLICE = FVM_GRID.cell_slice("aCL")
PEM_SLICE = FVM_GRID.cell_slice("PEM")
CCL_SLICE = FVM_GRID.cell_slice("cCL")
CGDL_SLICE = FVM_GRID.cell_slice("cGDL")

ANODE_POROUS_SLICE = slice(AGDL_SLICE.start, ACL_SLICE.stop)
CATHODE_POROUS_SLICE = slice(CCL_SLICE.start, CGDL_SLICE.stop)

AIR_DENSITY = Y_O2_IN * THERMAL_O2[0] + Y_N2_IN * THERMAL_N2[0]
AIR_HEAT_CAPACITY = Y_O2_IN * THERMAL_O2[1] + Y_N2_IN * THERMAL_N2[1]
AIR_CONDUCTIVITY = X_O2_IN * THERMAL_O2[2] + X_N2_IN * THERMAL_N2[2]

PHASE_DIRECTIONAL_DT = 1.0e-6       # s，用于代数相态闭合的方向导数


def _weighted_cell_average(values):
    values = np.asarray(values, dtype=float)
    return float(np.sum(values * DX_CELL) / np.sum(DX_CELL))


def _current_density_value(current_profile, time):
    value = current_profile(time) if callable(current_profile) else current_profile
    value = float(np.asarray(value, dtype=float))
    if not np.isfinite(value) or value < 0.0:
        raise ValueError("电流密度必须是非负有限标量")
    return value


def _porous_phase_fields(temperature, total_water, ice_mass):
    """仅在 GDL/CL 上应用蒸汽–液水–冰相态闭合；PEM 吸附水单独处理。"""
    vapor_mass = np.zeros(N_CELLS, dtype=float)
    liquid_mass = np.zeros(N_CELLS, dtype=float)
    ice_mass_effective = np.zeros(N_CELLS, dtype=float)
    gas_porosity = np.zeros(N_CELLS, dtype=float)
    liquid_fraction = np.zeros(N_CELLS, dtype=float)
    ice_fraction = np.zeros(N_CELLS, dtype=float)

    porous_ids = np.flatnonzero(POROUS_WATER_CELL_MASK)
    phase = WATER_MODEL.phase_state(
        total_water[porous_ids],
        ice_mass[porous_ids],
        temperature[porous_ids],
        POROSITY_DRY_CELL[porous_ids],
    )
    vapor_mass[porous_ids] = phase["vapor_mass"]
    liquid_mass[porous_ids] = phase["liquid_mass"]
    ice_mass_effective[porous_ids] = phase["ice_mass"]
    gas_porosity[porous_ids] = phase["gas_porosity"]
    liquid_fraction[porous_ids] = phase["liquid_fraction"]
    ice_fraction[porous_ids] = phase["ice_fraction"]

    return {
        "vapor_mass": vapor_mass,
        "liquid_mass": liquid_mass,
        "ice_mass_effective": ice_mass_effective,
        "gas_porosity": gas_porosity,
        "liquid_fraction": liquid_fraction,
        "ice_fraction": ice_fraction,
    }


def _thermal_property_fields(phase):
    volumetric_heat_capacity = np.zeros(N_CELLS, dtype=float)
    thermal_conductivity = np.zeros(N_CELLS, dtype=float)

    gas_properties = {
        "aGDL": THERMAL_H2,
        "aCL": THERMAL_H2,
        "PEM": (0.0, 0.0, 0.0),
        "cCL": (AIR_DENSITY, AIR_HEAT_CAPACITY, AIR_CONDUCTIVITY),
        "cGDL": (AIR_DENSITY, AIR_HEAT_CAPACITY, AIR_CONDUCTIVITY),
    }

    for region in REGION_TABLE:
        region_slice = region.cell_slice
        gas_density, gas_cp, gas_k = gas_properties[region.name]
        rho_cp, conductivity = ENERGY_MODEL.effective_properties(
            region.domain,
            phase["gas_porosity"][region_slice],
            phase["liquid_fraction"][region_slice],
            phase["ice_fraction"][region_slice],
            gas_density,
            gas_cp,
            gas_k,
        )
        volumetric_heat_capacity[region_slice] = rho_cp
        thermal_conductivity[region_slice] = conductivity

    volumetric_heat_capacity *= THERMAL_INERTIA_FACTOR
    return volumetric_heat_capacity, thermal_conductivity


def compute_model_fields(time, state, current_profile, case, boundary_conditions):
    """从五个 ODE 状态重构全部动态场、通量、源项、电压和时间导数。"""
    (
        temperature,
        c_h2,
        c_o2,
        total_water,
        ice_mass,
    ) = unpack_state(state)

    temperature_physical = np.maximum(temperature, 1.0)
    current_density = _current_density_value(current_profile, time)
    phase = _porous_phase_fields(
        temperature_physical,
        total_water,
        ice_mass,
    )

    # ------------------------------------------------------------------
    # 水：aCL--PEM--cCL 共用界面通量；PEM 内含反扩散和电渗拖曳。
    # ------------------------------------------------------------------
    water_diffusivity = np.zeros(N_CELLS, dtype=float)
    water_diffusivity[ANODE_POROUS_SLICE] = WATER_MODEL.vapor_effective_diffusivity(
        D_WATER_ANODE_REF,
        temperature_physical[ANODE_POROUS_SLICE],
        case.pressure,
        phase["gas_porosity"][ANODE_POROUS_SLICE],
    )
    water_diffusivity[CATHODE_POROUS_SLICE] = WATER_MODEL.vapor_effective_diffusivity(
        D_WATER_CATHODE_REF,
        temperature_physical[CATHODE_POROUS_SLICE],
        case.pressure,
        phase["gas_porosity"][CATHODE_POROUS_SLICE],
    )

    lambda_membrane = np.clip(
        WATER_MODEL.membrane_water_content(total_water[PEM_SLICE]),
        LAMBDA_MIN,
        LAMBDA_MAX,
    )
    membrane_water_diffusivity = WATER_MODEL.membrane_diffusivity(
        temperature_physical[PEM_SLICE],
        lambda_membrane,
    )
    water_diffusivity[PEM_SLICE] = membrane_water_diffusivity

    liquid_water_diffusivity = np.zeros(N_CELLS, dtype=float)
    for region_slice, permeability, contact_angle in (
        (AGDL_SLICE, PERM_GDL, CONTACT_ANGLE_GDL),
        (ACL_SLICE, PERM_CL, CONTACT_ANGLE_CL),
        (CCL_SLICE, PERM_CL, CONTACT_ANGLE_CL),
        (CGDL_SLICE, PERM_GDL, CONTACT_ANGLE_GDL),
    ):
        liquid_water_diffusivity[region_slice] = WATER_MODEL.liquid_capillary_diffusivity(
            phase["liquid_fraction"][region_slice],
            POROSITY_DRY_CELL[region_slice],
            permeability,
            contact_angle,
            temperature_physical[region_slice],
        )

    water_source = np.zeros(N_CELLS, dtype=float)
    water_source[CCL_SLICE] = CCL_DOMAIN.water_source(current_density)

    acl_interface_cell = ACL_SLICE.stop - 1
    pem_left_cell = PEM_SLICE.start
    pem_right_cell = PEM_SLICE.stop - 1
    ccl_interface_cell = CCL_SLICE.start
    left_interface_width = 0.5 * (
        DX_CELL[acl_interface_cell] + DX_CELL[pem_left_cell]
    )
    right_interface_width = 0.5 * (
        DX_CELL[pem_right_cell] + DX_CELL[ccl_interface_cell]
    )

    # 正值表示多孔层向膜吸水；换成全局 +x 通量时，右界面需反号。
    anode_to_membrane_diffusive_flux = WATER_MODEL.membrane_exchange_flux_to_membrane(
        phase["vapor_mass"][acl_interface_cell],
        phase["gas_porosity"][acl_interface_cell],
        phase["liquid_fraction"][acl_interface_cell],
        phase["ice_fraction"][acl_interface_cell],
        temperature_physical[acl_interface_cell],
        total_water[pem_left_cell],
        left_interface_width,
    )
    cathode_to_membrane_diffusive_flux = WATER_MODEL.membrane_exchange_flux_to_membrane(
        phase["vapor_mass"][ccl_interface_cell],
        phase["gas_porosity"][ccl_interface_cell],
        phase["liquid_fraction"][ccl_interface_cell],
        phase["ice_fraction"][ccl_interface_cell],
        temperature_physical[ccl_interface_cell],
        total_water[pem_right_cell],
        right_interface_width,
    )

    lambda_face = np.empty(lambda_membrane.size + 1, dtype=float)
    lambda_face[0] = lambda_membrane[0]
    lambda_face[-1] = lambda_membrane[-1]
    lambda_face[1:-1] = 0.5 * (
        lambda_membrane[:-1] + lambda_membrane[1:]
    )
    membrane_water_flux = WATER_MODEL.membrane_face_flux(
        total_water[PEM_SLICE],
        membrane_water_diffusivity,
        np.full(lambda_membrane.size + 1, current_density, dtype=float),
        lambda_face,
        DX_CELL[PEM_SLICE],
        left_flux=anode_to_membrane_diffusive_flux,
        right_flux=-cathode_to_membrane_diffusive_flux,
    )

    # 同一界面面通量同时进入相邻两个控制体，离散层面严格守恒。
    anode_vapor_flux = WATER_MODEL.porous_face_flux(
        phase["vapor_mass"][ANODE_POROUS_SLICE],
        water_diffusivity[ANODE_POROUS_SLICE],
        DX_CELL[ANODE_POROUS_SLICE],
        left_value=boundary_conditions.water_vapor_left,
    )
    anode_liquid_flux = WATER_MODEL.porous_face_flux(
        phase["liquid_mass"][ANODE_POROUS_SLICE],
        liquid_water_diffusivity[ANODE_POROUS_SLICE],
        DX_CELL[ANODE_POROUS_SLICE],
        left_value=0.0,
    )
    anode_water_flux = anode_vapor_flux + anode_liquid_flux
    anode_water_flux[-1] = membrane_water_flux[0]

    cathode_vapor_flux = WATER_MODEL.porous_face_flux(
        phase["vapor_mass"][CATHODE_POROUS_SLICE],
        water_diffusivity[CATHODE_POROUS_SLICE],
        DX_CELL[CATHODE_POROUS_SLICE],
        right_value=boundary_conditions.water_vapor_right,
    )
    cathode_liquid_flux = WATER_MODEL.porous_face_flux(
        phase["liquid_mass"][CATHODE_POROUS_SLICE],
        liquid_water_diffusivity[CATHODE_POROUS_SLICE],
        DX_CELL[CATHODE_POROUS_SLICE],
        right_value=0.0,
    )
    cathode_water_flux = cathode_vapor_flux + cathode_liquid_flux
    cathode_water_flux[0] = membrane_water_flux[-1]

    water_flux = np.zeros(N_FACES, dtype=float)
    water_flux[FACE_LEFT_BOUNDARY:FACE_ACL_PEM + 1] = anode_water_flux
    water_flux[FACE_ACL_PEM:FACE_PEM_CCL + 1] = membrane_water_flux
    water_flux[FACE_PEM_CCL:FACE_RIGHT_BOUNDARY + 1] = cathode_water_flux

    d_total_water_dt = np.zeros(N_CELLS, dtype=float)
    d_total_water_dt[ANODE_POROUS_SLICE] = WATER_MODEL.total_water_rhs(
        anode_water_flux,
        water_source[ANODE_POROUS_SLICE],
        DX_CELL[ANODE_POROUS_SLICE],
    )
    d_total_water_dt[PEM_SLICE] = WATER_MODEL.total_water_rhs(
        membrane_water_flux,
        water_source[PEM_SLICE],
        DX_CELL[PEM_SLICE],
    )
    d_total_water_dt[CATHODE_POROUS_SLICE] = WATER_MODEL.total_water_rhs(
        cathode_water_flux,
        water_source[CATHODE_POROUS_SLICE],
        DX_CELL[CATHODE_POROUS_SLICE],
    )

    ice_source = np.zeros(N_CELLS, dtype=float)
    porous_ids = np.flatnonzero(POROUS_WATER_CELL_MASK)
    raw_ice_source = WATER_MODEL.ice_source(
        phase["liquid_mass"][porous_ids],
        phase["ice_mass_effective"][porous_ids],
        temperature_physical[porous_ids],
    )

    # 冻结不能超过剩余孔隙容量；融化项保持原符号。
    remaining_ice_capacity = np.maximum(
        (
            POROSITY_DRY_CELL[porous_ids]
            - phase["liquid_fraction"][porous_ids]
            - phase["ice_fraction"][porous_ids]
        )
        * RHO_ICE,
        0.0,
    )
    raw_ice_source = np.where(
        raw_ice_source > 0.0,
        np.minimum(raw_ice_source, K_FREEZE_LIQUID * remaining_ice_capacity),
        raw_ice_source,
    )
    ice_source[porous_ids] = raw_ice_source
    d_ice_mass_dt = WATER_MODEL.ice_rhs(ice_source)

    # 由代数相态闭合沿当前 RHS 方向求 d(eps_g)/dt 和冷凝速率。
    total_water_probe = np.maximum(
        total_water + PHASE_DIRECTIONAL_DT * d_total_water_dt,
        0.0,
    )
    ice_mass_probe = np.maximum(
        phase["ice_mass_effective"] + PHASE_DIRECTIONAL_DT * d_ice_mass_dt,
        0.0,
    )
    phase_probe = _porous_phase_fields(
        temperature_physical,
        total_water_probe,
        ice_mass_probe,
    )
    raw_liquid_mass_rate = (
        phase_probe["liquid_mass"] - phase["liquid_mass"]
    ) / PHASE_DIRECTIONAL_DT
    raw_condensation_rate = raw_liquid_mass_rate + ice_source

    # 附件 1 给定的冷凝/蒸发动力学系数限制瞬时相平衡导数，避免液水初生点奇异。
    condensation_capacity = (
        K_CONDENSATION
        * np.maximum(phase_probe["liquid_mass"], phase["liquid_mass"])
    )
    evaporation_capacity = K_EVAPORATION * phase["liquid_mass"]
    condensation_rate = np.clip(
        raw_condensation_rate,
        -evaporation_capacity,
        condensation_capacity,
    )
    d_gas_porosity_dt = (
        -condensation_rate / RHO_LIQUID
        - ice_source / RHO_ICE
    )

    # ------------------------------------------------------------------
    # H2 / O2：只在各自传输域上计算，材料界面由面调和平均保证通量连续。
    # ------------------------------------------------------------------
    d_h2_effective = np.zeros(N_CELLS, dtype=float)
    d_o2_effective = np.zeros(N_CELLS, dtype=float)
    d_h2_effective[H2_CELL_MASK] = GAS_MODEL.effective_diffusivity(
        "h2",
        temperature_physical[H2_CELL_MASK],
        case.pressure,
        phase["gas_porosity"][H2_CELL_MASK],
    )
    d_o2_effective[O2_CELL_MASK] = GAS_MODEL.effective_diffusivity(
        "o2",
        temperature_physical[O2_CELL_MASK],
        case.pressure,
        phase["gas_porosity"][O2_CELL_MASK],
    )

    h2_source = np.zeros(N_CELLS, dtype=float)
    o2_source = np.zeros(N_CELLS, dtype=float)
    h2_source[ACL_SLICE] = ACL_DOMAIN.h2_source(current_density)
    o2_source[CCL_SLICE] = CCL_DOMAIN.o2_source(current_density)

    h2_flux = GAS_MODEL.face_flux(
        c_h2[H2_CELL_MASK],
        d_h2_effective[H2_CELL_MASK],
        DX_CELL[H2_CELL_MASK],
        left_value=boundary_conditions.h2_left_concentration(temperature_physical[0]),
        right_flux=boundary_conditions.h2_right_flux,
    )
    o2_flux = GAS_MODEL.face_flux(
        c_o2[O2_CELL_MASK],
        d_o2_effective[O2_CELL_MASK],
        DX_CELL[O2_CELL_MASK],
        left_flux=boundary_conditions.o2_left_flux,
        right_value=boundary_conditions.o2_right_concentration(temperature_physical[-1]),
    )

    d_c_h2_dt = np.zeros(N_CELLS, dtype=float)
    d_c_o2_dt = np.zeros(N_CELLS, dtype=float)
    d_c_h2_dt[H2_CELL_MASK] = GAS_MODEL.rhs(
        c_h2[H2_CELL_MASK],
        phase["gas_porosity"][H2_CELL_MASK],
        d_gas_porosity_dt[H2_CELL_MASK],
        h2_flux,
        h2_source[H2_CELL_MASK],
        DX_CELL[H2_CELL_MASK],
    )
    d_c_o2_dt[O2_CELL_MASK] = GAS_MODEL.rhs(
        c_o2[O2_CELL_MASK],
        phase["gas_porosity"][O2_CELL_MASK],
        d_gas_porosity_dt[O2_CELL_MASK],
        o2_flux,
        o2_source[O2_CELL_MASK],
        DX_CELL[O2_CELL_MASK],
    )

    # ------------------------------------------------------------------
    # 准稳态电荷、电压和三类损失。
    # ------------------------------------------------------------------
    electron_current, proton_current = CHARGE_MODEL.current_distribution(
        current_density,
        X_CELL,
    )
    current_residual = CHARGE_MODEL.residual(
        electron_current,
        proton_current,
        current_density,
    )

    active_area_factor_field = np.ones(N_CELLS, dtype=float)
    active_area_factor_field[CCL_SLICE] = CCL_DOMAIN.active_area_factor(
        phase["ice_fraction"][CCL_SLICE]
    )
    active_area_factor = float(np.mean(active_area_factor_field[CCL_SLICE]))

    mean_temperature = _weighted_cell_average(temperature_physical)
    p_h2_acl = float(np.mean(
        np.maximum(c_h2[ACL_SLICE], 0.0) * R * temperature_physical[ACL_SLICE]
    ))
    p_o2_ccl = float(np.mean(
        np.maximum(c_o2[CCL_SLICE], 0.0) * R * temperature_physical[CCL_SLICE]
    ))
    voltage = VOLTAGE_MODEL.compute(
        current_density=current_density,
        temperature_cell=mean_temperature,
        p_h2_acl=p_h2_acl,
        p_o2_ccl=p_o2_ccl,
        temperature_pem=temperature_physical[PEM_SLICE],
        lambda_pem=lambda_membrane,
        diffusivity_o2_cathode=d_o2_effective[CATHODE_POROUS_SLICE],
        concentration_o2_ccl=np.maximum(c_o2[CCL_SLICE], 0.0),
        active_area_factor=active_area_factor,
        dx_pem=DX_CELL[PEM_SLICE],
        dx_cathode=DX_CELL[CATHODE_POROUS_SLICE],
    )

    # ------------------------------------------------------------------
    # 能量：材料分区热物性、外边界 Robin 通量、电化学热和相变潜热。
    # ------------------------------------------------------------------
    volumetric_heat_capacity, thermal_conductivity = _thermal_property_fields(phase)
    left_heat_flux, right_heat_flux = ENERGY_MODEL.convection_boundary_flux(
        temperature_physical[0],
        temperature_physical[-1],
        boundary_conditions.ambient_temperature,
        boundary_conditions.convection_coefficient,
    )
    heat_flux = ENERGY_MODEL.face_heat_flux(
        temperature_physical,
        thermal_conductivity,
        DX_CELL,
        left_flux=left_heat_flux,
        right_flux=right_heat_flux,
    )
    electrochemical_heat_scalar = float(ENERGY_MODEL.electrochemical_heat(
        current_density,
        voltage["cell_voltage"],
    ))
    electrochemical_heat = np.full(N_CELLS, electrochemical_heat_scalar, dtype=float)
    phase_change_heat = ENERGY_MODEL.phase_change_heat(
        ice_source,
        condensation_rate,
    )
    auxiliary_heat = np.zeros(N_CELLS, dtype=float)
    d_temperature_dt = ENERGY_MODEL.temperature_rhs(
        volumetric_heat_capacity,
        heat_flux,
        electrochemical_heat,
        phase_change_heat,
        auxiliary_heat,
        DX_CELL,
    )

    water_balance_residual = float(
        np.sum(d_total_water_dt * DX_CELL)
        - (
            water_flux[FACE_LEFT_BOUNDARY]
            - water_flux[FACE_RIGHT_BOUNDARY]
            + np.sum(water_source * DX_CELL)
        )
    )

    return {
        "time": float(time),
        "current_density": current_density,
        "temperature": temperature,
        "c_h2": c_h2,
        "c_o2": c_o2,
        "total_water": total_water,
        "ice_mass": ice_mass,
        "vapor_mass": phase["vapor_mass"],
        "liquid_mass": phase["liquid_mass"],
        "gas_porosity": phase["gas_porosity"],
        "liquid_fraction": phase["liquid_fraction"],
        "ice_fraction": phase["ice_fraction"],
        "lambda_membrane": lambda_membrane,
        "d_h2_effective": d_h2_effective,
        "d_o2_effective": d_o2_effective,
        "water_diffusivity": water_diffusivity,
        "liquid_water_diffusivity": liquid_water_diffusivity,
        "d_gas_porosity_dt": d_gas_porosity_dt,
        "raw_condensation_rate": raw_condensation_rate,
        "condensation_rate": condensation_rate,
        "ice_source": ice_source,
        "h2_source": h2_source,
        "o2_source": o2_source,
        "water_source": water_source,
        "h2_flux": h2_flux,
        "o2_flux": o2_flux,
        "water_flux": water_flux,
        "membrane_water_flux": membrane_water_flux,
        "anode_to_membrane_diffusive_flux": anode_to_membrane_diffusive_flux,
        "cathode_to_membrane_diffusive_flux": cathode_to_membrane_diffusive_flux,
        "electron_current": electron_current,
        "proton_current": proton_current,
        "current_residual": current_residual,
        "active_area_factor_field": active_area_factor_field,
        "active_area_factor": active_area_factor,
        "p_h2_acl": p_h2_acl,
        "p_o2_ccl": p_o2_ccl,
        "voltage": voltage,
        "volumetric_heat_capacity": volumetric_heat_capacity,
        "thermal_conductivity": thermal_conductivity,
        "heat_flux": heat_flux,
        "electrochemical_heat": electrochemical_heat,
        "phase_change_heat": phase_change_heat,
        "d_temperature_dt": d_temperature_dt,
        "d_c_h2_dt": d_c_h2_dt,
        "d_c_o2_dt": d_c_o2_dt,
        "d_total_water_dt": d_total_water_dt,
        "d_ice_mass_dt": d_ice_mass_dt,
        "water_balance_residual": water_balance_residual,
    }


def model_rhs(time, state, current_profile, case, boundary_conditions):
    fields = compute_model_fields(
        time,
        state,
        current_profile,
        case,
        boundary_conditions,
    )
    return pack_rhs(
        fields["d_temperature_dt"],
        fields["d_c_h2_dt"],
        fields["d_c_o2_dt"],
        fields["d_total_water_dt"],
        fields["d_ice_mass_dt"],
    )


In [8]:
# Cell 8：读取附件 2，建立两条实验加载曲线并用稀疏 BDF 求解
import sys
from pathlib import Path

LOCAL_NOTEBOOK_DEPS = Path.cwd() / ".codex_nbdeps"
if LOCAL_NOTEBOOK_DEPS.exists() and str(LOCAL_NOTEBOOK_DEPS) not in sys.path:
    sys.path.insert(0, str(LOCAL_NOTEBOOK_DEPS))

import pandas as pd
from scipy.integrate import solve_ivp
from scipy.sparse.linalg import spsolve
from scipy.sparse import coo_matrix, diags, identity, lil_matrix


def locate_attachment2():
    preferred = (
        Path.cwd()
        / "氢燃料电池低温冷启动建模与控制策略研究  附件"
        / "附件2.xlsx"
    )
    if preferred.exists():
        return preferred

    candidates = [
        path
        for path in Path.cwd().glob("**/附件2.xlsx")
        if not path.name.startswith("~$")
    ]
    if len(candidates) != 1:
        raise FileNotFoundError("无法唯一定位附件2.xlsx")
    return candidates[0]


@dataclass(frozen=True)
class ExperimentalTrace:
    label: str
    data: pd.DataFrame

    def __post_init__(self):
        required = {
            "time_s",
            "current_A",
            "voltage_V",
            "temperature_C",
            "current_density_A_cm2",
        }
        if set(self.data.columns) != required:
            raise ValueError(f"{self.label}: 附件2列结构不符合预期")
        if self.data.empty or self.data.isna().any().any():
            raise ValueError(f"{self.label}: 附件2存在空值")
        if np.any(np.diff(self.data["time_s"].to_numpy()) <= 0.0):
            raise ValueError(f"{self.label}: 时间必须严格递增")

    @property
    def time(self):
        return self.data["time_s"].to_numpy(dtype=float)

    @property
    def current_density_si(self):
        return (
            self.data["current_density_A_cm2"].to_numpy(dtype=float)
            * CURRENT_A_CM2_TO_A_M2
        )

    def current_profile(self, time):
        return np.interp(
            time,
            self.time,
            self.current_density_si,
            left=self.current_density_si[0],
            right=self.current_density_si[-1],
        )


def load_attachment2(path):
    """按原始行序读取附件2；统一列名但不修改原始数值。"""
    workbook = pd.ExcelFile(path)
    if len(workbook.sheet_names) < 2:
        raise ValueError("附件2至少应包含 -20℃ 和 -25℃ 两个工作表")

    traces = {}
    for label, sheet_name in zip(("-20C", "-25C"), workbook.sheet_names[:2]):
        frame = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=None,
            skiprows=2,
            usecols="A:E",
        )
        frame.columns = [
            "time_s",
            "current_A",
            "voltage_V",
            "temperature_C",
            "current_density_A_cm2",
        ]
        frame = frame.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
        traces[label] = ExperimentalTrace(label=label, data=frame)
    return traces


def build_jacobian_sparsity():
    """构造局部近邻耦合的稀疏 Jacobian 结构，避免 12948×12948 稠密差分。"""
    sparsity = lil_matrix((N_STATE, N_STATE), dtype=np.int8)

    h2_state_by_cell = np.full(N_CELLS, -1, dtype=np.int32)
    o2_state_by_cell = np.full(N_CELLS, -1, dtype=np.int32)
    h2_state_by_cell[H2_STATE_CELL_IDS] = STATE_H2.start + np.arange(N_H2)
    o2_state_by_cell[O2_STATE_CELL_IDS] = STATE_O2.start + np.arange(N_O2)

    def neighbors(cell):
        return range(max(0, cell - 1), min(N_CELLS, cell + 2))

    for cell in range(N_CELLS):
        t_row = STATE_T.start + cell
        mw_row = STATE_MW.start + cell
        mi_row = STATE_MI.start + cell

        for neighbor in neighbors(cell):
            t_col = STATE_T.start + neighbor
            mw_col = STATE_MW.start + neighbor
            mi_col = STATE_MI.start + neighbor
            sparsity[t_row, (t_col, mw_col, mi_col)] = 1
            sparsity[mw_row, (t_col, mw_col, mi_col)] = 1

            h2_col = h2_state_by_cell[neighbor]
            o2_col = o2_state_by_cell[neighbor]
            if h2_col >= 0:
                sparsity[t_row, h2_col] = 1
            if o2_col >= 0:
                sparsity[t_row, o2_col] = 1

        sparsity[mi_row, (
            STATE_T.start + cell,
            STATE_MW.start + cell,
            STATE_MI.start + cell,
        )] = 1

        h2_row = h2_state_by_cell[cell]
        if h2_row >= 0:
            for neighbor in neighbors(cell):
                h2_col = h2_state_by_cell[neighbor]
                if h2_col >= 0:
                    sparsity[h2_row, h2_col] = 1
                sparsity[h2_row, STATE_T.start + neighbor] = 1
                sparsity[h2_row, STATE_MW.start + neighbor] = 1
                sparsity[h2_row, STATE_MI.start + neighbor] = 1

        o2_row = o2_state_by_cell[cell]
        if o2_row >= 0:
            for neighbor in neighbors(cell):
                o2_col = o2_state_by_cell[neighbor]
                if o2_col >= 0:
                    sparsity[o2_row, o2_col] = 1
                sparsity[o2_row, STATE_T.start + neighbor] = 1
                sparsity[o2_row, STATE_MW.start + neighbor] = 1
                sparsity[o2_row, STATE_MI.start + neighbor] = 1

    return sparsity.tocsr()



def model_jacobian(time, state, current_profile, case, boundary_conditions):
    """主要扩散/导热项的解析稀疏 Jacobian；弱全局耦合由 Newton 迭代修正。"""
    fields = compute_model_fields(
        time,
        state,
        current_profile,
        case,
        boundary_conditions,
    )
    rows = []
    cols = []
    values = []

    def add(row, col, value):
        if value != 0.0 and np.isfinite(value):
            rows.append(int(row))
            cols.append(int(col))
            values.append(float(value))

    def add_diffusion_block(
        state_start,
        cell_ids,
        diffusivity,
        storage,
        left_dirichlet=False,
        right_dirichlet=False,
        left_robin=0.0,
        right_robin=0.0,
    ):
        cell_ids = np.asarray(cell_ids, dtype=np.int32)
        diffusivity = np.maximum(np.asarray(diffusivity, dtype=float), EPS_FLOOR)
        storage = np.maximum(np.asarray(storage, dtype=float), EPS_FLOOR)
        widths = DX_CELL[cell_ids]
        count = cell_ids.size
        if count == 0:
            return

        if count > 1:
            face_diffusivity = _weighted_harmonic_face(diffusivity, widths)
            center_distance = 0.5 * (widths[:-1] + widths[1:])
            for local in range(count - 1):
                right_rate = (
                    face_diffusivity[local]
                    / center_distance[local]
                    / widths[local]
                    / storage[local]
                )
                left_rate = (
                    face_diffusivity[local]
                    / center_distance[local]
                    / widths[local + 1]
                    / storage[local + 1]
                )
                left_state = state_start + local
                right_state = state_start + local + 1
                add(left_state, left_state, -right_rate)
                add(left_state, right_state, right_rate)
                add(right_state, left_state, left_rate)
                add(right_state, right_state, -left_rate)

        if left_dirichlet:
            rate = diffusivity[0] / (0.5 * widths[0] ** 2 * storage[0])
            add(state_start, state_start, -rate)
        if right_dirichlet:
            rate = diffusivity[-1] / (0.5 * widths[-1] ** 2 * storage[-1])
            add(state_start + count - 1, state_start + count - 1, -rate)
        if left_robin > 0.0:
            rate = left_robin / (widths[0] * storage[0])
            add(state_start, state_start, -rate)
        if right_robin > 0.0:
            rate = right_robin / (widths[-1] * storage[-1])
            add(state_start + count - 1, state_start + count - 1, -rate)

    all_ids = ALL_STATE_CELL_IDS
    add_diffusion_block(
        STATE_T.start,
        all_ids,
        fields["thermal_conductivity"],
        fields["volumetric_heat_capacity"],
        left_robin=boundary_conditions.convection_coefficient,
        right_robin=boundary_conditions.convection_coefficient,
    )

    add_diffusion_block(
        STATE_H2.start,
        H2_STATE_CELL_IDS,
        fields["d_h2_effective"][H2_CELL_MASK],
        fields["gas_porosity"][H2_CELL_MASK],
        left_dirichlet=True,
    )
    add_diffusion_block(
        STATE_O2.start,
        O2_STATE_CELL_IDS,
        fields["d_o2_effective"][O2_CELL_MASK],
        fields["gas_porosity"][O2_CELL_MASK],
        right_dirichlet=True,
    )

    # 总水扩散只对未饱和蒸汽区有一阶灵敏度；饱和区液水为代数储库。
    for region_slice, left_dirichlet, right_dirichlet in (
        (ANODE_POROUS_SLICE, True, False),
        (CATHODE_POROUS_SLICE, False, True),
    ):
        cell_ids = np.arange(region_slice.start, region_slice.stop, dtype=np.int32)
        region_total_water = fields["total_water"][region_slice]
        region_ice_mass = np.maximum(fields["ice_mass"][region_slice], 0.0)
        region_temperature = fields["temperature"][region_slice]
        region_porosity = POROSITY_DRY_CELL[region_slice]
        vapor_saturation = WATER_MODEL.saturated_vapor_density(region_temperature)
        pore_without_liquid = np.maximum(
            region_porosity - region_ice_mass / RHO_ICE,
            EPS_FLOOR,
        )
        saturation_excess = (
            np.maximum(region_total_water, 0.0)
            - region_ice_mass
            - vapor_saturation * pore_without_liquid
        )
        transition_ratio = np.clip(
            saturation_excess / PHASE_SMOOTHING_MASS,
            0.0,
            1.0,
        )
        positive_part_derivative = np.where(
            saturation_excess <= 0.0,
            0.0,
            np.where(
                saturation_excess >= PHASE_SMOOTHING_MASS,
                1.0,
                4.0 * transition_ratio - 3.0 * transition_ratio**2,
            ),
        )
        phase_denominator = np.maximum(
            1.0 - vapor_saturation / RHO_LIQUID,
            EPS_FLOOR,
        )
        vapor_mobility = np.clip(
            1.0 - positive_part_derivative / phase_denominator,
            0.0,
            1.0,
        )
        add_diffusion_block(
            STATE_MW.start + region_slice.start,
            cell_ids,
            (
                fields["water_diffusivity"][region_slice] * vapor_mobility
                + fields["liquid_water_diffusivity"][region_slice]
                * (1.0 - vapor_mobility)
            ),
            np.ones(cell_ids.size),
            left_dirichlet=left_dirichlet,
            right_dirichlet=right_dirichlet,
        )

    pem_cell_ids = np.arange(PEM_SLICE.start, PEM_SLICE.stop, dtype=np.int32)
    add_diffusion_block(
        STATE_MW.start + PEM_SLICE.start,
        pem_cell_ids,
        fields["water_diffusivity"][PEM_SLICE],
        np.ones(pem_cell_ids.size),
    )

    temperature = fields["temperature"]
    ice_source = fields["ice_source"]
    for cell in np.flatnonzero(POROUS_WATER_CELL_MASK):
        # clip(·, 0, pore_capacity) 的平坦区导数为 0；只在线性相变支路加导数。
        if abs(ice_source[cell]) <= 1.0e-14:
            continue
        ice_row = STATE_MI.start + cell
        if temperature[cell] < T_FREEZE:
            add(ice_row, STATE_MI.start + cell, -K_FREEZE_LIQUID)
            add(ice_row, STATE_MW.start + cell, K_FREEZE_LIQUID)
        else:
            add(ice_row, STATE_MI.start + cell, -K_MELT_ICE)

    return coo_matrix(
        (values, (rows, cols)),
        shape=(N_STATE, N_STATE),
        dtype=float,
    ).tocsr()

def state_absolute_tolerance():
    """按状态量尺度设置容差，避免零水/零冰状态的数值 Jacobian 溢出。"""
    tolerance = np.full(N_STATE, 1.0e-5, dtype=float)
    tolerance[STATE_T] = 1.0e-4
    tolerance[STATE_H2] = 1.0e-5
    tolerance[STATE_O2] = 1.0e-5
    tolerance[STATE_MW] = 1.0e-5
    tolerance[STATE_MI] = 1.0e-5
    return tolerance


@dataclass
class IntegratedTrajectory:
    """逐实验时间区间积分后，与 solve_ivp.OdeResult 兼容的轻量结果。"""

    t: np.ndarray
    y: np.ndarray
    success: bool
    message: str
    nfev: int
    njev: int
    nlu: int


@dataclass
class ColdStartSolution:
    label: str
    case: ColdStartCase
    boundary_conditions: ColdStartBoundaryConditions
    experiment: ExperimentalTrace
    result: object


_SPLIT_DYNAMIC_ROW_MASK = np.ones(N_STATE, dtype=float)
_SPLIT_DYNAMIC_ROW_MASK[STATE_MW] = 0.0
_SPLIT_DYNAMIC_ROW_MASK[STATE_MI] = 0.0
_SPLIT_DYNAMIC_ROW_SELECTOR = diags(_SPLIT_DYNAMIC_ROW_MASK, format="csr")
_STATE_IDENTITY = identity(N_STATE, format="csr")
_WATER_IDENTITY = identity(N_MW, format="csr")


def _transport_rhs(time, state, current_profile, case, boundary_conditions):
    """算子分裂的气体/能量子问题；本子步固定总水和冰。"""
    fields = compute_model_fields(
        time, state, current_profile, case, boundary_conditions
    )
    zeros = np.zeros(N_CELLS, dtype=float)
    d_c_h2_dt = np.zeros(N_CELLS, dtype=float)
    d_c_o2_dt = np.zeros(N_CELLS, dtype=float)
    d_c_h2_dt[H2_CELL_MASK] = GAS_MODEL.rhs(
        fields["c_h2"][H2_CELL_MASK],
        fields["gas_porosity"][H2_CELL_MASK],
        zeros[H2_CELL_MASK],
        fields["h2_flux"],
        fields["h2_source"][H2_CELL_MASK],
        DX_CELL[H2_CELL_MASK],
    )
    d_c_o2_dt[O2_CELL_MASK] = GAS_MODEL.rhs(
        fields["c_o2"][O2_CELL_MASK],
        fields["gas_porosity"][O2_CELL_MASK],
        zeros[O2_CELL_MASK],
        fields["o2_flux"],
        fields["o2_source"][O2_CELL_MASK],
        DX_CELL[O2_CELL_MASK],
    )
    d_temperature_dt = (
        fields["d_temperature_dt"]
        - fields["phase_change_heat"] / fields["volumetric_heat_capacity"]
    )
    return pack_rhs(
        d_temperature_dt,
        d_c_h2_dt,
        d_c_o2_dt,
        zeros,
        zeros,
    )


def _transport_jacobian(time, state, current_profile, case, boundary_conditions):
    return _SPLIT_DYNAMIC_ROW_SELECTOR @ model_jacobian(
        time, state, current_profile, case, boundary_conditions
    )


def _advance_split_step(
    time_left,
    time_right,
    state,
    current_profile,
    case,
    boundary_conditions,
    rtol,
):
    """气体/导热隐式，水扩散线性隐式，相变与潜热显式。"""
    dt = float(time_right - time_left)
    transport_rate = _transport_rhs(
        time_left, state, current_profile, case, boundary_conditions
    )
    transport_matrix = _transport_jacobian(
        time_left, state, current_profile, case, boundary_conditions
    )
    transport_increment = spsolve(
        _STATE_IDENTITY - dt * transport_matrix,
        dt * transport_rate,
    )
    if not np.all(np.isfinite(transport_increment)):
        raise RuntimeError("线性隐式气体/导热子步出现非有限增量")
    state_new = state + transport_increment

    fields = compute_model_fields(
        time_right, state_new, current_profile, case, boundary_conditions
    )
    full_jacobian = model_jacobian(
        time_right, state_new, current_profile, case, boundary_conditions
    )
    water_jacobian = full_jacobian[STATE_MW, STATE_MW]
    water_increment = spsolve(
        _WATER_IDENTITY - dt * water_jacobian,
        dt * fields["d_total_water_dt"],
    )

    # 用各区域真实容量限制隐式水更新；不再用单一反应源上限截断 PEM 重分布。
    water_before = np.maximum(state_new[STATE_MW].copy(), 0.0)
    water_increment = np.maximum(water_increment, -water_before)
    water_updated = water_before + water_increment

    porous_capacity = POROSITY_DRY_CELL * RHO_LIQUID
    water_updated[POROUS_WATER_CELL_MASK] = np.clip(
        water_updated[POROUS_WATER_CELL_MASK],
        0.0,
        porous_capacity[POROUS_WATER_CELL_MASK],
    )
    pem_water_min = float(WATER_MODEL.membrane_water_mass(LAMBDA_MIN))
    pem_water_max = float(WATER_MODEL.membrane_water_mass(LAMBDA_MAX))
    water_updated[PEM_SLICE] = np.clip(
        water_updated[PEM_SLICE],
        pem_water_min,
        pem_water_max,
    )
    state_new[STATE_MW] = water_updated

    phase_fields = compute_model_fields(
        time_right, state_new, current_profile, case, boundary_conditions
    )
    state_new[STATE_MI] = np.clip(
        state_new[STATE_MI] + dt * phase_fields["d_ice_mass_dt"],
        0.0,
        state_new[STATE_MW],
    )
    state_new[STATE_T] += (
        dt
        * phase_fields["phase_change_heat"]
        / phase_fields["volumetric_heat_capacity"]
    )
    state_new[STATE_T] = np.clip(state_new[STATE_T], 150.0, 400.0)
    state_new[STATE_H2] = np.maximum(state_new[STATE_H2], 0.0)
    state_new[STATE_O2] = np.maximum(state_new[STATE_O2], 0.0)
    return state_new


def solve_cold_start_case(
    experiment,
    initial_temperature_celsius,
    jacobian_sparsity,
    rtol=1.0e-3,
    max_step=0.2,
):
    """按实验时间区间作 Strang 型物理分裂，输出时刻与附件 2 完全一致。"""
    temperature_kelvin = initial_temperature_celsius + 273.15
    case = ColdStartCase(
        initial_temperature=temperature_kelvin,
        ambient_temperature=temperature_kelvin,
    )
    boundary_conditions = ColdStartBoundaryConditions.from_case(case)
    initial = build_initial_conditions(case)
    state = pack_state(
        initial.temperature,
        initial.c_h2,
        initial.c_o2,
        initial.total_water,
        initial.ice_mass,
    )

    times = np.asarray(experiment.time, dtype=float)
    states = np.empty((N_STATE, times.size), dtype=float)
    states[:, 0] = state
    nfev = 0
    njev = 0
    nlu = 0

    for index in range(times.size - 1):
        interval_left = float(times[index])
        interval_right = float(times[index + 1])
        interval_length = interval_right - interval_left
        substep_count = max(1, int(np.ceil(interval_length / max_step)))
        substep_times = np.linspace(interval_left, interval_right, substep_count + 1)
        for time_left, time_right in zip(substep_times[:-1], substep_times[1:]):
            try:
                state = _advance_split_step(
                    time_left,
                    time_right,
                    state,
                    experiment.current_profile,
                    case,
                    boundary_conditions,
                    rtol,
                )
            except RuntimeError as error:
                raise RuntimeError(
                    f"{experiment.label} 在 {time_left:.3f}--{time_right:.3f} s "
                    f"求解失败：{error}"
                ) from error
            nfev += 1
            njev += 1
            nlu += 2
        states[:, index + 1] = state

    result = IntegratedTrajectory(
        t=times,
        y=states,
        success=True,
        message="分裂隐式有限体积积分成功",
        nfev=nfev,
        njev=njev,
        nlu=nlu,
    )
    return ColdStartSolution(
        label=experiment.label,
        case=case,
        boundary_conditions=boundary_conditions,
        experiment=experiment,
        result=result,
    )


ATTACHMENT2_PATH = locate_attachment2()
EXPERIMENT_TRACES = load_attachment2(ATTACHMENT2_PATH)
JACOBIAN_SPARSITY = build_jacobian_sparsity()

SIMULATION_RESULTS = {
    "-20C": solve_cold_start_case(
        EXPERIMENT_TRACES["-20C"],
        initial_temperature_celsius=-20.0,
        jacobian_sparsity=JACOBIAN_SPARSITY,
    ),
    "-25C": solve_cold_start_case(
        EXPERIMENT_TRACES["-25C"],
        initial_temperature_celsius=-25.0,
        jacobian_sparsity=JACOBIAN_SPARSITY,
    ),
}


In [9]:
# ============================================================
# Cell 9：后处理、附件 2 对比、动态变量导出与 300 dpi 作图
# ============================================================

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager


OUTPUT_DIR = Path.cwd() / "outputs" / "porous_battery_q1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Latin 字形优先 Times New Roman；缺失的中文字形回退到微软雅黑。
mpl.rcParams.update({
    "font.family": ["Times New Roman", "Microsoft YaHei"],
    "axes.unicode_minus": False,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.7,
})


def _space_average(values, mask=None):
    values = np.asarray(values, dtype=float)
    if mask is None:
        mask = np.ones(N_CELLS, dtype=bool)
    weights = DX_CELL[mask]
    return float(np.sum(values[mask] * weights) / np.sum(weights))


def _safe_r2(observed, predicted):
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.sum((observed - np.mean(observed)) ** 2)
    if denominator <= EPS_FLOOR:
        return np.nan
    return float(1.0 - np.sum((observed - predicted) ** 2) / denominator)


def _save_figure(figure, filename):
    output_path = OUTPUT_DIR / filename
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(figure)
    return output_path


def build_case_outputs(simulation):
    """在每个附件采样时刻重构动态派生场，并生成最终空间场。"""
    rows = []
    final_fields = None
    result = simulation.result
    experiment = simulation.experiment

    for index, time_value in enumerate(result.t):
        fields = compute_model_fields(
            float(time_value),
            result.y[:, index],
            experiment.current_profile,
            simulation.case,
            simulation.boundary_conditions,
        )
        final_fields = fields
        voltage = fields["voltage"]
        porous_mask = POROUS_WATER_CELL_MASK

        rows.append({
            "case": simulation.label,
            "time_s": float(time_value),
            "current_A": float(experiment.data.iloc[index]["current_A"]),
            "current_density_A_cm2": fields["current_density"] / CURRENT_A_CM2_TO_A_M2,
            "temperature_experiment_C": float(experiment.data.iloc[index]["temperature_C"]),
            "temperature_model_C": _space_average(fields["temperature"]) - 273.15,
            "voltage_experiment_V": float(experiment.data.iloc[index]["voltage_V"]),
            "voltage_model_V": voltage["cell_voltage"],
            "temperature_min_C": float(np.min(fields["temperature"]) - 273.15),
            "temperature_max_C": float(np.max(fields["temperature"]) - 273.15),
            "ice_fraction_average": _space_average(fields["ice_fraction"]),
            "ice_fraction_max": float(np.max(fields["ice_fraction"])),
            "ice_fraction_ccl_average": _space_average(fields["ice_fraction"], CCL_SLICE),
            "liquid_fraction_max": float(np.max(fields["liquid_fraction"])),
            "gas_porosity_min": float(np.min(fields["gas_porosity"][porous_mask])),
            "gas_porosity_ccl_average": _space_average(fields["gas_porosity"], CCL_SLICE),
            "d_h2_effective_average_m2_s": _space_average(fields["d_h2_effective"], H2_CELL_MASK),
            "d_o2_effective_average_m2_s": _space_average(fields["d_o2_effective"], O2_CELL_MASK),
            "d_o2_effective_ccl_average_m2_s": _space_average(fields["d_o2_effective"], CCL_SLICE),
            "d_water_cathode_average_m2_s": _space_average(fields["water_diffusivity"], CATHODE_POROUS_SLICE),
            "d_liquid_capillary_cathode_average_m2_s": _space_average(
                fields["liquid_water_diffusivity"], CATHODE_POROUS_SLICE
            ),
            "lambda_pem_average": float(np.mean(fields["lambda_membrane"])),
            "lambda_pem_min": float(np.min(fields["lambda_membrane"])),
            "lambda_pem_max": float(np.max(fields["lambda_membrane"])),
            "membrane_water_average_kg_m3": float(
                np.mean(fields["total_water"][PEM_SLICE])
            ),
            "anode_pem_water_flux_kg_m2_s": float(
                fields["water_flux"][FACE_ACL_PEM]
            ),
            "pem_cathode_water_flux_kg_m2_s": float(
                fields["water_flux"][FACE_PEM_CCL]
            ),
            "active_area_factor": fields["active_area_factor"],
            "reversible_voltage_V": voltage["reversible_voltage"],
            "activation_loss_V": voltage["activation_loss"],
            "ohmic_loss_V": voltage["ohmic_loss"],
            "concentration_loss_V": voltage["concentration_loss"],
            "limiting_current_A_m2": voltage["limiting_current"],
            "exchange_current_density_A_m2": voltage["exchange_current_density_A_m2"],
            "hydration_activity_factor": voltage["hydration_activity_factor"],
            "effective_exchange_current_density_A_m2": voltage[
                "effective_exchange_current_density_A_m2"
            ],
            "transport_limited": voltage["transport_limited"],
            "electrochemical_heat_average_W_m3": _space_average(fields["electrochemical_heat"]),
            "phase_change_heat_average_W_m3": _space_average(fields["phase_change_heat"]),
            "current_residual_max_A_m2": float(np.max(np.abs(fields["current_residual"]))),
            "water_balance_residual_kg_m2_s": fields["water_balance_residual"],
        })

    summary = pd.DataFrame(rows)
    summary["temperature_error_C"] = (
        summary["temperature_model_C"] - summary["temperature_experiment_C"]
    )
    summary["voltage_error_V"] = (
        summary["voltage_model_V"] - summary["voltage_experiment_V"]
    )

    spatial = pd.DataFrame({
        "x_um": X_CELL * 1.0e6,
        "region_id": REGION_ID,
        "region": [DOMAINS[int(region_id)].name for region_id in REGION_ID],
        "temperature_C": final_fields["temperature"] - 273.15,
        "c_h2_mol_m3": final_fields["c_h2"],
        "c_o2_mol_m3": final_fields["c_o2"],
        "total_water_kg_m3": final_fields["total_water"],
        "vapor_water_kg_m3": final_fields["vapor_mass"],
        "liquid_water_kg_m3": final_fields["liquid_mass"],
        "ice_water_kg_m3": final_fields["ice_mass"],
        "liquid_fraction": final_fields["liquid_fraction"],
        "ice_fraction": final_fields["ice_fraction"],
        "gas_porosity": final_fields["gas_porosity"],
        "d_h2_effective_m2_s": final_fields["d_h2_effective"],
        "d_o2_effective_m2_s": final_fields["d_o2_effective"],
        "d_water_effective_m2_s": final_fields["water_diffusivity"],
        "d_liquid_capillary_m2_s": final_fields["liquid_water_diffusivity"],
        "thermal_conductivity_W_m_K": final_fields["thermal_conductivity"],
        "volumetric_heat_capacity_J_m3_K": final_fields["volumetric_heat_capacity"],
    })
    spatial["temperature_smoothed_C"] = spatial.groupby(
        "region", sort=False
    )["temperature_C"].transform(
        lambda series: series.rolling(
            window=51,
            center=True,
            min_periods=1,
        ).mean()
    )
    return summary, spatial


CASE_OUTPUTS = {}
for case_label, simulation in SIMULATION_RESULTS.items():
    summary_frame, spatial_frame = build_case_outputs(simulation)
    CASE_OUTPUTS[case_label] = {
        "summary": summary_frame,
        "spatial_final": spatial_frame,
    }
    summary_frame.to_csv(
        OUTPUT_DIR / f"{case_label}_model_vs_attachment2.csv",
        index=False,
        encoding="utf-8-sig",
    )
    spatial_frame.to_csv(
        OUTPUT_DIR / f"{case_label}_final_spatial_fields.csv",
        index=False,
        encoding="utf-8-sig",
    )


error_rows = []
for case_label, output in CASE_OUTPUTS.items():
    frame = output["summary"]
    for variable, observed_column, model_column, unit in (
        ("temperature", "temperature_experiment_C", "temperature_model_C", "degC"),
        ("voltage", "voltage_experiment_V", "voltage_model_V", "V"),
    ):
        observed = frame[observed_column].to_numpy(dtype=float)
        predicted = frame[model_column].to_numpy(dtype=float)
        error = predicted - observed
        error_rows.append({
            "case": case_label,
            "variable": variable,
            "unit": unit,
            "MAE": float(np.mean(np.abs(error))),
            "RMSE": float(np.sqrt(np.mean(error**2))),
            "max_absolute_error": float(np.max(np.abs(error))),
            "bias": float(np.mean(error)),
            "R2": _safe_r2(observed, predicted),
        })

ERROR_SUMMARY = pd.DataFrame(error_rows)
ERROR_SUMMARY.to_csv(
    OUTPUT_DIR / "error_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

raw_attachment2 = pd.concat(
    [trace.data.assign(case=label) for label, trace in EXPERIMENT_TRACES.items()],
    ignore_index=True,
)
raw_attachment2.to_csv(
    OUTPUT_DIR / "attachment2_normalized_raw_data.csv",
    index=False,
    encoding="utf-8-sig",
)


# 1. 温度和电压：模型与附件 2 同时间轴对比
figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex="col")
for row, case_label in enumerate(("-20C", "-25C")):
    frame = CASE_OUTPUTS[case_label]["summary"]
    case_title = "-20 ℃工况" if case_label == "-20C" else "-25 ℃工况"
    axes[row, 0].plot(frame["time_s"], frame["temperature_experiment_C"], "o", ms=2.5, label="附件2")
    axes[row, 0].plot(frame["time_s"], frame["temperature_model_C"], label="模型")
    axes[row, 0].set_ylabel("平均温度 / ℃")
    axes[row, 0].set_title(f"{case_title}：温度")
    axes[row, 0].grid(alpha=0.25)
    axes[row, 0].legend()
    axes[row, 1].plot(frame["time_s"], frame["voltage_experiment_V"], "o", ms=2.5, label="附件2")
    axes[row, 1].plot(frame["time_s"], frame["voltage_model_V"], label="模型")
    axes[row, 1].set_ylabel("单电池电压 / V")
    axes[row, 1].set_title(f"{case_title}：电压")
    axes[row, 1].grid(alpha=0.25)
    axes[row, 1].legend()
axes[-1, 0].set_xlabel("时间 / s")
axes[-1, 1].set_xlabel("时间 / s")
figure.suptitle("一维冷启动模型与附件2实验数据对比")
figure.tight_layout()
FIGURE_COMPARISON = _save_figure(figure, "figure_1_model_vs_attachment2.png")


# 2. 冰堵、气相孔隙率和催化活性面积
figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex="col")
for row, case_label in enumerate(("-20C", "-25C")):
    frame = CASE_OUTPUTS[case_label]["summary"]
    case_title = "-20 ℃" if case_label == "-20C" else "-25 ℃"
    axes[row, 0].plot(frame["time_s"], frame["ice_fraction_max"], label="最大冰体积分数")
    axes[row, 0].plot(frame["time_s"], frame["ice_fraction_ccl_average"], label="cCL平均冰体积分数")
    axes[row, 0].set_ylabel("冰体积分数 / -")
    axes[row, 0].set_title(f"{case_title}：冰生成")
    axes[row, 0].grid(alpha=0.25)
    axes[row, 0].legend()
    axes[row, 1].plot(frame["time_s"], frame["gas_porosity_min"], label="最小气相孔隙率")
    axes[row, 1].plot(frame["time_s"], frame["gas_porosity_ccl_average"], label="cCL平均气相孔隙率")
    axes[row, 1].plot(frame["time_s"], frame["active_area_factor"], label="有效反应面积因子")
    axes[row, 1].set_ylabel("孔隙率或面积因子 / -")
    axes[row, 1].set_title(f"{case_title}：冰堵反馈")
    axes[row, 1].grid(alpha=0.25)
    axes[row, 1].legend()
axes[-1, 0].set_xlabel("时间 / s")
axes[-1, 1].set_xlabel("时间 / s")
figure.suptitle("低温冷启动中的水–冰相变与孔隙堵塞")
figure.tight_layout()
FIGURE_ICE = _save_figure(figure, "figure_2_ice_and_porosity.png")


# 3. 有效扩散系数和三类电压损失
figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex="col")
for row, case_label in enumerate(("-20C", "-25C")):
    frame = CASE_OUTPUTS[case_label]["summary"]
    case_title = "-20 ℃" if case_label == "-20C" else "-25 ℃"
    axes[row, 0].plot(frame["time_s"], frame["d_h2_effective_average_m2_s"], label=r"$D_{H_2,eff}$")
    axes[row, 0].plot(frame["time_s"], frame["d_o2_effective_average_m2_s"], label=r"$D_{O_2,eff}$")
    axes[row, 0].plot(frame["time_s"], frame["d_water_cathode_average_m2_s"], label=r"$D_{v,eff}$")
    axes[row, 0].plot(
        frame["time_s"],
        frame["d_liquid_capillary_cathode_average_m2_s"],
        label=r"$D_{l,cap}$",
    )
    axes[row, 0].set_yscale("log")
    axes[row, 0].set_ylabel("有效扩散系数 / m²·s⁻¹")
    axes[row, 0].set_title(f"{case_title}：传质能力（对数坐标）")
    axes[row, 0].grid(alpha=0.25)
    axes[row, 0].legend()
    axes[row, 1].plot(frame["time_s"], frame["activation_loss_V"], label="活化损失")
    axes[row, 1].plot(frame["time_s"], frame["ohmic_loss_V"], label="欧姆损失")
    axes[row, 1].plot(frame["time_s"], frame["concentration_loss_V"], label="浓差损失")
    axes[row, 1].set_ylabel("电压损失 / V")
    axes[row, 1].set_title(f"{case_title}：损失分解")
    axes[row, 1].grid(alpha=0.25)
    axes[row, 1].legend()
axes[-1, 0].set_xlabel("时间 / s")
axes[-1, 1].set_xlabel("时间 / s")
figure.suptitle("动态有效扩散系数与单电池电压损失")
figure.tight_layout()
FIGURE_TRANSPORT = _save_figure(figure, "figure_3_diffusivity_and_losses.png")


# 4. 末时刻沿厚度方向的空间分布
figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex="col")
for row, case_label in enumerate(("-20C", "-25C")):
    frame = CASE_OUTPUTS[case_label]["spatial_final"]
    case_title = "-20 ℃" if case_label == "-20C" else "-25 ℃"
    axes[row, 0].plot(frame["x_um"], frame["temperature_smoothed_C"], label="温度（51单元滑动平均）")
    axes[row, 0].set_ylabel("温度 / ℃")
    axes[row, 0].set_title(f"{case_title}：末时刻温度场")
    axes[row, 0].grid(alpha=0.25)
    axes[row, 1].plot(frame["x_um"], frame["ice_fraction"], label="冰体积分数")
    axes[row, 1].plot(frame["x_um"], frame["gas_porosity"], label="气相孔隙率")
    axes[row, 1].set_ylabel("体积分数 / -")
    axes[row, 1].set_title(f"{case_title}：末时刻相分布")
    axes[row, 1].grid(alpha=0.25)
    axes[row, 1].legend()
axes[-1, 0].set_xlabel("厚度坐标 / μm")
axes[-1, 1].set_xlabel("厚度坐标 / μm")
figure.suptitle("膜电极厚度方向的最终空间场")
figure.tight_layout()
FIGURE_SPATIAL = _save_figure(figure, "figure_4_final_spatial_fields.png")


# 5. PEM 含水量与跨界面水通量，用于验证新增水输运闭环
figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex="col")
for row, case_label in enumerate(("-20C", "-25C")):
    frame = CASE_OUTPUTS[case_label]["summary"]
    case_title = "-20 ℃" if case_label == "-20C" else "-25 ℃"
    axes[row, 0].plot(frame["time_s"], frame["lambda_pem_average"], label="PEM平均 λ")
    axes[row, 0].fill_between(
        frame["time_s"],
        frame["lambda_pem_min"],
        frame["lambda_pem_max"],
        alpha=0.20,
        label="PEM空间范围",
    )
    axes[row, 0].set_ylabel("膜含水量 λ / -")
    axes[row, 0].set_title(f"{case_title}：PEM动态水合")
    axes[row, 0].grid(alpha=0.25)
    axes[row, 0].legend()

    time_values = frame["time_s"].to_numpy(dtype=float)
    anode_flux = frame["anode_pem_water_flux_kg_m2_s"].to_numpy(dtype=float)
    cathode_flux = frame["pem_cathode_water_flux_kg_m2_s"].to_numpy(dtype=float)
    time_step = np.diff(time_values)
    anode_transfer = np.zeros_like(time_values)
    cathode_transfer = np.zeros_like(time_values)
    anode_transfer[1:] = np.cumsum(
        0.5 * (anode_flux[1:] + anode_flux[:-1]) * time_step
    )
    cathode_transfer[1:] = np.cumsum(
        0.5 * (cathode_flux[1:] + cathode_flux[:-1]) * time_step
    )
    axes[row, 1].plot(
        time_values,
        anode_transfer,
        label="aCL→PEM累计传水量",
    )
    axes[row, 1].plot(
        time_values,
        cathode_transfer,
        label="PEM/cCL累计传水量（+x）",
    )
    axes[row, 1].axhline(0.0, color="black", lw=0.7, alpha=0.5)
    axes[row, 1].set_ylabel("累计传水量 / kg·m⁻²")
    axes[row, 1].set_title(f"{case_title}：PEM界面累计水交换")
    axes[row, 1].grid(alpha=0.25)
    axes[row, 1].legend()
axes[-1, 0].set_xlabel("时间 / s")
axes[-1, 1].set_xlabel("时间 / s")
figure.suptitle("PEM含水输运：反扩散、电渗拖曳与界面交换")
figure.tight_layout()
FIGURE_PEM_WATER = _save_figure(figure, "figure_5_pem_water_transport.png")


print(f"输出目录：{OUTPUT_DIR}")
print("误差汇总：")
print(ERROR_SUMMARY.to_string(index=False))


输出目录：E:\math\math\outputs\porous_battery_q1
误差汇总：
case    variable unit      MAE     RMSE  max_absolute_error      bias       R2
-20C temperature degC 0.032173 0.040988            0.073735 -0.030852 0.999733
-20C     voltage    V 0.016899 0.026342            0.070814  0.009676 0.918821
-25C temperature degC 0.134137 0.201907            0.478207  0.133844 0.993677
-25C     voltage    V 0.023257 0.026458            0.051589 -0.008857 0.919154
